# ปฏิบัติการที่ 12 การประมวลผลภาพและ CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NKSR22/Introduction_to_AI_Labs/blob/main/lab12-computer-vision.ipynb)

ปฏิบัติการประกอบหนังสือ *ปัญญาประดิษฐ์เบื้องต้น* (นครินทร์ ศรีปัญญา) บทที่ 6 คอมพิวเตอร์วิทัศน์: AI กับภาพ หัวข้อ 6.1 ภาพดิจิทัล 6.2 การประมวลผลภาพเบื้องต้น 6.3 การจำแนกภาพด้วยการเรียนรู้เชิงลึก 6.4 การตรวจจับวัตถุและการแบ่งส่วนภาพ และ 6.5 การประยุกต์ในอุตสาหกรรม

## วัตถุประสงค์

เมื่อทำปฏิบัติการนี้จบ ผู้เรียนสามารถ

1. แทนภาพระดับเทาและภาพสีด้วยอาร์เรย์ NumPy อ่านพิกัดแบบ (แถว, คอลัมน์) ได้ถูกต้องเมื่อใช้ร่วมกับ OpenCV และคำนวณขนาดข้อมูลภาพกับ GSD ของกล้องได้
2. แปลงภาพเป็นระดับเทาและ HSV ด้วยสูตรและด้วย `cv2.cvtColor` (ลำดับ BGR, ค่า H 0–179) และอธิบายว่าเหตุใดกฎใน HSV จึงทนต่อเงา
3. เขียนฟังก์ชันกรองภาพเองตามรหัสเทียมในหนังสือ เทียบกับ `cv2.filter2D` `cv2.blur` `cv2.GaussianBlur` `cv2.medianBlur` และเลือกตัวกรองให้ตรงกับชนิดของสัญญาณรบกวนได้
4. หาขอบด้วย Sobel และ Canny หาค่าเกณฑ์ด้วยวิธีของ Otsu และใช้การเปิด การปิด และองค์ประกอบเชื่อมต่อแปลงภาพเป็นวัตถุที่นับและวัดขนาดได้
5. เปรียบเทียบการจำแนกภาพด้วยคุณลักษณะที่ออกแบบเอง กับ "ส่วนหลักที่แช่แข็ง + ส่วนหัวที่ฝึก" ซึ่งเป็นแนวคิดของ Transfer Learning นับพารามิเตอร์ที่ต้องฝึก และรายงานเมทริกซ์ความสับสนพร้อมตัววัดรายคลาสได้
6. เขียนฟังก์ชัน IoU, ถอดรหัส YOLO, NMS และ Average Precision เอง ตรวจกับตัวอย่างในหนังสือ และวัดผลตัวตรวจจับวัตถุอย่างง่ายบนภาพจำลองได้
7. วัดผลการแบ่งส่วนภาพด้วย IoU และ Dice และนำผลจากภาพไปตัดสินใจทางวิศวกรรม เช่น ภาพถ่ายความร้อนของขั้วเบรกเกอร์ได้

## เนื้อหาในหนังสือที่เกี่ยวข้อง

- 6.1.1 พิกเซลและภาพในรูปเมทริกซ์ — ตัวอย่างที่ 6.1–6.2
- 6.1.2 ช่องสีและปริภูมิสี — ตัวอย่างที่ 6.3
- 6.2.1 การกรองภาพและ Convolution — ตัวอย่างที่ 6.4
- 6.2.2 การหาขอบภาพ — ตัวอย่างที่ 6.5–6.6
- 6.2.3 Thresholding และ Morphology — ตัวอย่างที่ 6.7–6.9
- 6.3.1 จาก Feature แบบดั้งเดิมสู่ CNN — ตัวอย่างที่ 6.10–6.11
- 6.3.2 การใช้ Transfer Learning กับชุดข้อมูลขนาดเล็ก — ตัวอย่างที่ 6.12–6.13
- 6.4.1 Object Detection และแนวคิดของ YOLO — ตัวอย่างที่ 6.14–6.17
- 6.4.2 Image Segmentation — ตัวอย่างที่ 6.18–6.19
- 6.5 การประยุกต์ในอุตสาหกรรม — ตัวอย่างที่ 6.20–6.22

## ความรู้พื้นฐานที่ควรมี

- Python, NumPy และ matplotlib ระดับปฏิบัติการที่ 1 และ 9 (ทบทวนได้ในภาคผนวก ก)
- ตัววัดผลของงานจำแนกจากหัวข้อ 4.3.1 (ปฏิบัติการที่ 9) และแนวคิดของ CNN จากหัวข้อ 5.5 (ปฏิบัติการที่ 11)
- อ่านบทที่ 6 มาก่อน notebook นี้คำนวณตัวอย่างเชิงตัวเลขของบทซ้ำด้วยโค้ด แล้วขยายไปสู่ภาพจำลองที่ใหญ่ขึ้น

## วิธีใช้ notebook นี้

- กดปุ่ม **Open in Colab** ด้านบนเพื่อเปิดใน Google Colab แล้วรันทีละเซลล์จากบนลงล่างด้วย `Shift + Enter`
- ถ้ารันเซลล์ข้ามลำดับแล้วเกิดข้อผิดพลาด ให้เลือกเมนู **Runtime → Run all** เพื่อรันใหม่ทั้งหมด
- เซลล์ที่มี `# TODO` ในส่วน "แบบฝึกปฏิบัติ" คือส่วนที่ผู้เรียนต้องเติมโค้ดเอง
- ภาพทั้งหมดเป็นภาพจำลองที่สร้างด้วยโค้ด (seed = 42) จึงไม่ต้องดาวน์โหลดข้อมูล ทั้ง notebook รันบน CPU ได้ในเวลาไม่ถึง 1 นาที
- ส่วนที่ทำเครื่องหมาย **(ทางเลือก)** ใช้โมเดลที่ฝึกกับ ImageNet/COCO ไว้แล้ว ต้องใช้ PyTorch และอินเทอร์เน็ตสำหรับดาวน์โหลดน้ำหนัก (Colab มีให้พร้อม ควรเปิด GPU ที่ **Runtime → Change runtime type**) ส่วนนี้ **ปิดไว้เป็นค่าเริ่มต้น** ด้วยตัวแปร `RUN_PRETRAINED = False` เปิดเมื่อพร้อม

| ส่วน | หัวข้อ | ตัวอย่างในหนังสือ |
|:---:|---|---|
| 1 | ภาพในรูปอาร์เรย์ ขนาดข้อมูล GSD และปริภูมิสี | 6.1–6.3 |
| 2 | การกรองภาพ: เขียน FILTER2D เองเทียบกับ OpenCV | 6.4 |
| 3 | การหาขอบ: Sobel, Canny และฮิสเทอรีซิส | 6.5–6.6 |
| 4 | Otsu, สัณฐานวิทยา และองค์ประกอบเชื่อมต่อบนภาพหน้าจอมิเตอร์ | 6.7–6.9 |
| 5 | การจำแนกภาพ: คุณลักษณะที่ออกแบบเอง เทียบกับส่วนหลักที่แช่แข็ง + ส่วนหัว และ Transfer Learning (ทางเลือก) | 6.10–6.13 |
| 6 | การตรวจจับวัตถุและการแบ่งส่วนภาพ: IoU, YOLO, ตัวตรวจจับอย่างง่าย, IoU/Dice | 6.14–6.19 |
| 7 | การประยุกต์: เกณฑ์ความผิดปกติ อัตราฐาน และภาพถ่ายความร้อน | 6.20–6.22 |
| 8 | แบบฝึกปฏิบัติ: Otsu, สัณฐานวิทยา, NMS และ AP ที่เขียนเอง | 6.7, 6.8, 6.16, 6.17 |

## เตรียมเครื่องมือ

ใช้ `numpy` `pandas` `matplotlib` `scipy` `scikit-learn` และ **OpenCV** (`cv2`) ซึ่ง Google Colab มีให้แล้วทั้งหมด ถ้ารันบนเครื่องอื่นที่ยังไม่มี OpenCV เซลล์ถัดไปจะติดตั้ง `opencv-python-headless` ให้อัตโนมัติ (ต้องใช้อินเทอร์เน็ตครั้งแรกครั้งเดียว)

เซลล์ถัดไปยังสร้างข้อมูลขนาดเล็กของบทที่ 6 ไว้ใช้ตลอด notebook

- `CRACK_PATCH` — ตัวอย่างที่ 6.1 บริเวณ 6 × 6 พิกเซลของผิวลูกถ้วยที่มีรอยร้าว
- `METAL` — ตัวอย่างที่ 6.4 ผิวชิ้นงานโลหะ 5 × 5 ที่มีจุดรบกวน 255 ที่ (1, 3)
- `BOX_KERNEL` `GAUSS_KERNEL` `SOBEL_X` `SOBEL_Y` — เคอร์เนลในตารางที่ 6.2 และหัวข้อ 6.2.2

> **หมายเหตุ** ป้ายกำกับบนกราฟใช้ภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ภาษาไทยติดตั้งไว้ พิกัดภาพในข้อความเขียนเป็น (แถว, คอลัมน์) นับจาก 0 ที่มุมซ้ายบน ตามข้อตกลงของหนังสือ

In [ ]:
import math
import subprocess
import sys

try:
    import cv2
except ImportError:  # Colab มี OpenCV อยู่แล้ว ส่วนนี้สำหรับเครื่องที่ยังไม่มี
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy import ndimage
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42  # กำหนด seed ให้ภาพจำลองและผลการสุ่มเหมือนเดิมทุกครั้งที่รัน
plt.rcParams["figure.dpi"] = 100
COLORS = {"blue": "#1F4E79", "orange": "#E07B39", "green": "#3A7D44", "gray": "#6B6B6B"}
print(f"OpenCV {cv2.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")


def compare(label, got, expected, tol):
    """พิมพ์ว่าค่าที่คำนวณตรงกับค่าในหนังสือ (ภายในความคลาดเคลื่อน tol) หรือไม่"""
    mark = "✓ ตรงกับหนังสือ" if abs(got - expected) <= tol else f"✗ ไม่ตรง (หนังสือ: {expected})"
    print(f"{label}: {got:.4f}  {mark}")


def show_matrix(matrix, title, ax, vmin=0, vmax=255, fmt="{:.0f}", cmap="gray"):
    """แสดงเมทริกซ์ขนาดเล็กเป็นภาพ พร้อมเขียนค่าของทุกพิกเซล"""
    matrix = np.asarray(matrix)
    ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    for (r, c), value in np.ndenumerate(matrix):
        shade = (value - vmin) / (vmax - vmin + 1e-9)
        ax.text(c, r, fmt.format(value), ha="center", va="center", fontsize=8,
                color="black" if shade > 0.5 else "white")
    ax.set_title(title)
    ax.set_xlabel("column c")
    ax.set_ylabel("row r")


def print_binary(binary, title=""):
    """พิมพ์ภาพสองค่าด้วย ■ (1) และ □ (0) แบบเดียวกับหนังสือ"""
    if title:
        print(title)
    for row in np.asarray(binary):
        print(" ".join("■" if v else "□" for v in row))


# ตัวอย่างที่ 6.1 บริเวณ 6 × 6 ของผิวลูกถ้วยที่มีรอยร้าว (ภาพระดับเทา 8 บิต)
CRACK_PATCH = np.array([
    [182, 185, 180, 70, 178, 181],
    [184, 183, 75, 72, 180, 183],
    [181, 179, 68, 177, 182, 180],
    [186, 80, 65, 181, 179, 184],
    [183, 78, 176, 180, 185, 182],
    [180, 74, 179, 183, 181, 186],
], dtype=np.uint8)

# ตัวอย่างที่ 6.4 ผิวโลหะที่มีจุดรบกวนสีขาวที่ (1, 3)
METAL = np.array([
    [100, 102, 98, 101, 99],
    [101, 99, 100, 255, 100],
    [98, 100, 103, 99, 101],
    [102, 97, 100, 100, 98],
    [99, 101, 99, 102, 100],
], dtype=np.uint8)

BOX_KERNEL = np.ones((3, 3)) / 9                                  # ตัวกรองค่าเฉลี่ย
GAUSS_KERNEL = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16    # ตัวกรองเกาส์เซียน 3 × 3
SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
SOBEL_Y = SOBEL_X.T

## ส่วนที่ 1 ภาพดิจิทัลในรูปอาร์เรย์

(หนังสือหัวข้อ 6.1)

### 1.1 อ่านภาพรอยร้าวในตัวอย่างที่ 6.1

ภาพระดับเทา 8 บิตคืออาร์เรย์สองมิติชนิด `uint8` ที่มีค่า 0–255 เข้าถึงพิกเซลด้วย `F[r, c]` (แถวก่อน คอลัมน์หลัง) เซลล์ถัดไปทำตามขั้นที่ 1–3 ของตัวอย่างที่ 6.1 แล้วแสดงภาพพร้อมค่า

In [ ]:
F = CRACK_PATCH
print("ขนาด (สูง, กว้าง):", F.shape, "| ชนิดข้อมูล:", F.dtype)
print("f(0, 3) =", F[0, 3], "| f(5, 5) =", F[5, 5])
print("ผลรวมแต่ละแถว:", F.sum(axis=1), "| ผลรวมทั้งหมด:", F.sum())
r_min, c_min = np.unravel_index(F.argmin(), F.shape)
print(f"ค่าต่ำสุด {F.min()} ที่ (r, c) = ({r_min}, {c_min}) | ค่าสูงสุด {F.max()}")
compare("ค่าเฉลี่ยทั้งบริเวณ", F.mean(), 157.3, 0.05)

crack_mask = F < 100                       # ค่าเกณฑ์ T = 100 (รอยร้าวมืดกว่าผิว)
print("\nพิกเซลรอยร้าว (r, c):", [tuple(int(v) for v in p) for p in np.argwhere(crack_mask)])
print("ค่า:", F[crack_mask], "| จำนวน", crack_mask.sum())
compare("ค่าเฉลี่ยรอยร้าว", F[crack_mask].mean(), 72.75, 1e-9)
compare("ค่าเฉลี่ยผิวปกติ (28 พิกเซล)", F[~crack_mask].mean(), 181.4, 0.05)
compare("ความเปรียบต่าง", F[~crack_mask].mean() - F[crack_mask].mean(), 108.7, 0.05)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
show_matrix(F, "Example 6.1: gray levels", axes[0])
show_matrix(crack_mask.astype(int), "pixels < 100 (crack)", axes[1], vmax=1)
plt.tight_layout()
plt.show()

**ข้อควรระวัง — ลำดับพิกัด** NumPy ใช้ `F[r, c]` แต่ฟังก์ชันวาดรูปของ OpenCV รับจุดเป็น $(x, y) = (c, r)$ และ `cv2.resize` รับขนาดเป็น (กว้าง, สูง) เซลล์ถัดไปแสดงผลของการใส่ลำดับผิด

In [ ]:
canvas = np.zeros((6, 10), dtype=np.uint8)   # สูง 6 แถว กว้าง 10 คอลัมน์
r, c = 1, 7

by_numpy = canvas.copy()
by_numpy[r, c] = 255                                        # NumPy: [แถว, คอลัมน์]
by_opencv = canvas.copy()
cv2.rectangle(by_opencv, (c, r), (c, r), 255, -1)           # OpenCV: (x, y) = (คอลัมน์, แถว)
swapped = canvas.copy()
cv2.rectangle(swapped, (r, c), (r, c), 255, -1)             # ใส่ (แถว, คอลัมน์) ผิดลำดับ

for name, img in [("NumPy F[1, 7]", by_numpy), ("OpenCV (x=7, y=1)", by_opencv), ("OpenCV ใส่ผิดเป็น (1, 7)", swapped)]:
    print(f"{name:26s} → พิกเซลที่สว่าง (r, c): {[tuple(int(v) for v in p) for p in np.argwhere(img)]}")

print("\ncv2.resize(canvas, (20, 12)).shape =", cv2.resize(canvas, (20, 12)).shape, "← dsize คือ (กว้าง, สูง) แต่ shape คือ (สูง, กว้าง)")

จุดที่ใส่ผิดลำดับตกนอกภาพ (แถว 7 ในภาพที่มีเพียง 6 แถว) จึงหายไปเงียบ ๆ โดยไม่มีข้อความแจ้งข้อผิดพลาด นี่คือสาเหตุของ "กรอบที่วาดอยู่ผิดตำแหน่ง" ที่พบบ่อยที่สุด

### 1.2 ขนาดข้อมูลภาพและ GSD (ตัวอย่างที่ 6.2)

$\text{GSD} = D \cdot s_w / (f \cdot N_w)$ คือขนาดจริงบนวัตถุที่พิกเซลหนึ่งแทน ฟังก์ชันด้านล่างคำนวณตารางระยะถ่ายของกล้องโดรนในตัวอย่างที่ 6.2 และกราฟความกว้างของรอยร้าว 2 mm ในหน่วยพิกเซล

In [ ]:
def image_bytes(width, height, channels=1, bits_per_channel=8):
    """จำนวนไบต์ของภาพที่ไม่บีบอัด"""
    return width * height * channels * bits_per_channel // 8


def gsd_mm(distance_mm, sensor_width_mm, focal_mm, pixels_wide):
    """ระยะชักตัวอย่างบนพื้น (mm ต่อพิกเซล) จากแบบจำลองกล้องรูเข็ม"""
    return distance_mm * sensor_width_mm / (focal_mm * pixels_wide)


def max_distance_mm(target_gsd_mm, sensor_width_mm, focal_mm, pixels_wide):
    """ระยะถ่ายไกลสุดที่ยังได้ GSD ไม่เกินค่าที่ต้องการ"""
    return target_gsd_mm * focal_mm * pixels_wide / sensor_width_mm


gray_full_hd = image_bytes(1920, 1080)
color_full_hd = image_bytes(1920, 1080, channels=3)
print(f"ภาพระดับเทา 1920×1080: {gray_full_hd:,} ไบต์ | ภาพสี: {color_full_hd:,} ไบต์ | วิดีโอสี 30 fps: {color_full_hd * 30 / 1e6:.0f} MB/s")

DRONE_CAMERA = dict(sensor_width_mm=12.0, focal_mm=24.0, pixels_wide=6000)
rows = []
for distance_m in (5, 8, 10, 15, 20):
    g = gsd_mm(distance_m * 1000, **DRONE_CAMERA)
    rows.append({"distance (m)": distance_m, "GSD (mm/px)": g, "2-mm crack (px)": 2 / g})
display(pd.DataFrame(rows).set_index("distance (m)").round(3))
compare("GSD ที่ 10 m (mm/px)", gsd_mm(10_000, **DRONE_CAMERA), 0.833, 5e-4)
compare("ระยะไกลสุดเมื่อต้องการ ≥ 3 px (m)", max_distance_mm(2 / 3, **DRONE_CAMERA) / 1000, 8, 1e-9)
compare("ระยะไกลสุดเมื่อใช้เลนส์ 48 mm (m)", max_distance_mm(2 / 3, 12.0, 48.0, 6000) / 1000, 16, 1e-9)

distances = np.linspace(2, 25, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for focal, color in [(24, COLORS["blue"]), (48, COLORS["orange"])]:
    widths = 2 / gsd_mm(distances * 1000, 12.0, focal, 6000)
    ax.plot(distances, widths, color=color, label=f"f = {focal} mm")
ax.axhline(3, color=COLORS["gray"], ls="--", label="3-pixel requirement")
ax.set_xlabel("distance to insulator (m)")
ax.set_ylabel("2-mm crack width (pixels)")
ax.set_ylim(0, 12)
ax.legend()
ax.set_title("Example 6.2: resolution needed to see a crack")
plt.show()

**คำถามชวนคิด** ถ้าข้อกำหนดความปลอดภัยบังคับให้โดรนอยู่ห่างสายส่งอย่างน้อย 12 m ทีมงานมีทางเลือกอะไรบ้าง ลองคำนวณด้วยฟังก์ชัน `max_distance_mm` ว่าต้องใช้เลนส์ยาวเท่าใด

### 1.3 ปริภูมิสี: ระดับเทาและ HSV (ตัวอย่างที่ 6.3)

เขียนสูตร BT.601 และสูตร HSV ของหนังสือเอง แล้วเทียบกับ `cv2.cvtColor` ข้อควรระวังสองข้อของ OpenCV คือ (1) ภาพที่อ่านด้วย `cv2.imread` เก็บช่องสีเป็นลำดับ **BGR** ไม่ใช่ RGB และ (2) ภาพ 8 บิตเก็บ H เป็น 0–179 (องศาหารสอง) และ S, V เป็น 0–255

In [ ]:
def rgb_to_gray(r, g, b):
    """ระดับเทาตามมาตรฐาน ITU-R BT.601"""
    return 0.299 * r + 0.587 * g + 0.114 * b


def rgb_to_hsv_deg(r, g, b):
    """แปลง RGB (0–255) เป็น H (องศา) S, V (0–1) ตามสูตรในหัวข้อ 6.1.2"""
    r, g, b = r / 255, g / 255, b / 255
    mx, mn = max(r, g, b), min(r, g, b)
    delta = mx - mn
    s = delta / mx if mx > 0 else 0.0
    if delta == 0:
        h = 0.0                                  # สีเทา: H ไม่มีความหมาย
    elif mx == r:
        h = 60 * (((g - b) / delta) % 6)
    elif mx == g:
        h = 60 * ((b - r) / delta + 2)
    else:
        h = 60 * ((r - g) / delta + 4)
    return h, s, mx


RUST_SUN, RUST_SHADE = (224, 123, 57), (112, 62, 29)
table = pd.DataFrame(
    [[*px, rgb_to_gray(*px), *rgb_to_hsv_deg(*px)] for px in (RUST_SUN, RUST_SHADE)],
    columns=["R", "G", "B", "gray", "H (deg)", "S", "V"], index=["rust, sunlit", "rust, shade"])
display(table.round(3))
compare("H สนิมโดนแดด", table.loc["rust, sunlit", "H (deg)"], 23.7, 0.05)
compare("H สนิมในเงา", table.loc["rust, shade", "H (deg)"], 23.9, 0.05)

# OpenCV: สร้างภาพ 1 × 2 พิกเซลแบบ RGB แล้วแปลงเป็น BGR ตามที่ OpenCV คาดหวัง
pixels_rgb = np.array([[RUST_SUN, RUST_SHADE]], dtype=np.uint8)
pixels_bgr = cv2.cvtColor(pixels_rgb, cv2.COLOR_RGB2BGR)
gray_cv = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2GRAY)[0]
hsv_cv = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2HSV)[0]
wrong_gray = cv2.cvtColor(pixels_rgb, cv2.COLOR_BGR2GRAY)[0]   # ลืมว่า OpenCV ใช้ BGR
print("\ncv2 ระดับเทา:", gray_cv, "(หนังสือ 146, 73) | ถ้าป้อน RGB ผิดเป็น BGR ได้:", wrong_gray)
print("cv2 HSV ดิบ (H 0–179, S, V 0–255):", hsv_cv.tolist())
print("แปลงกลับ: H =", hsv_cv[:, 0] * 2, "องศา | S =", (hsv_cv[:, 1] / 255).round(3), "| V =", (hsv_cv[:, 2] / 255).round(3))

ค่า H ของ OpenCV ถูกปัดเป็นจำนวนเต็มหลังหารสอง (24° ทั้งคู่) จึงหยาบกว่าสูตรเล็กน้อย ถ้าลืมสลับ BGR ระดับเทาของพิกเซลสนิมจะผิดไปราว 30 ระดับโดยไม่มีข้อความเตือน

ต่อไปขยายเป็นภาพจำลองของขาเสาโครงเหล็กสีเทาที่มีคราบสนิม โดยครึ่งขวาอยู่ในเงา (แสงเหลือครึ่งหนึ่ง) แล้วเปรียบเทียบกฎสองแบบในขั้นที่ 4 ของตัวอย่างที่ 6.3 (เพิ่มเงื่อนไข $V > 0.15$ ตามคำแนะนำในหนังสือ)

In [ ]:
def make_rust_scene(seed=RANDOM_SEED, height=120, width=160):
    """ภาพ RGB จำลองของเหล็กทาสีเทาที่มีคราบสนิม ครึ่งขวาอยู่ในเงา คืน (ภาพ, หน้ากากสนิมจริง)"""
    rng = np.random.default_rng(seed)
    image = np.ones((height, width, 3)) * np.array([150.0, 150.0, 146.0])
    rust_mask = np.zeros((height, width), dtype=np.uint8)
    for _ in range(10):
        center = (int(rng.integers(10, width - 10)), int(rng.integers(10, height - 10)))
        axes = (int(rng.integers(4, 14)), int(rng.integers(3, 9)))
        cv2.ellipse(rust_mask, center, axes, float(rng.uniform(0, 180)), 0, 360, 1, -1)
    image[rust_mask == 1] = np.array(RUST_SUN) * rng.uniform(0.9, 1.1, size=(rust_mask.sum(), 1))
    image += rng.normal(0, 5, image.shape)
    light = np.ones((height, width))
    light[:, width // 2:] = 0.5                       # เงาพาดครึ่งขวาของภาพ
    image *= light[..., None]
    return np.clip(image, 0, 255).astype(np.uint8), rust_mask.astype(bool)


rust_rgb, rust_truth = make_rust_scene()
R, G, B = [rust_rgb[..., k].astype(int) for k in range(3)]
rule_rgb = (R > 150) & (B < 100)

hsv = cv2.cvtColor(rust_rgb, cv2.COLOR_RGB2HSV)       # ภาพนี้เป็น RGB จึงใช้ COLOR_RGB2HSV
H_deg, S, V = hsv[..., 0] * 2.0, hsv[..., 1] / 255, hsv[..., 2] / 255
rule_hsv = (H_deg >= 10) & (H_deg <= 35) & (S > 0.5) & (V > 0.15)

half = rust_rgb.shape[1] // 2
rows = []
for name, pred in [("RGB rule", rule_rgb), ("HSV rule", rule_hsv)]:
    recall_sun = (pred & rust_truth)[:, :half].sum() / rust_truth[:, :half].sum()
    recall_shade = (pred & rust_truth)[:, half:].sum() / rust_truth[:, half:].sum()
    false_pos = (pred & ~rust_truth).sum()
    rows.append({"rule": name, "recall (sunlit half)": recall_sun, "recall (shaded half)": recall_shade,
                 "false-positive pixels": false_pos})
display(pd.DataFrame(rows).set_index("rule").round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].imshow(rust_rgb)
axes[0].set_title("simulated tower leg (right half in shadow)")
axes[1].imshow(rule_rgb, cmap="gray")
axes[1].set_title("RGB rule: R > 150 and B < 100")
axes[2].imshow(rule_hsv, cmap="gray")
axes[2].set_title("HSV rule: 10 <= H <= 35, S > 0.5")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**คำถามชวนคิด** กฎ HSV แยกสนิมออกจากเหล็กสีเทาได้ แต่ถ้าเสาบางต้นทาสีส้มเพื่อความปลอดภัยทางการบิน กฎนี้จะเกิดปัญหาอะไร และนี่เป็นแรงจูงใจของหัวข้อ 6.3 อย่างไร

## ส่วนที่ 2 การกรองภาพ

(หนังสือหัวข้อ 6.2.1)

### 2.1 เขียน FILTER2D เองตามรหัสเทียม

ฟังก์ชัน `filter2d` ด้านล่างแปลรหัสเทียม FILTER2D ในหนังสือตรงตัว คือเติมขอบภาพ แล้ววนทุกพิกเซล คูณเคอร์เนลกับเพื่อนบ้านและรวมกัน (แบบสหสัมพันธ์) ส่วน `median_filter` ใช้ค่ามัธยฐานของเพื่อนบ้านแทนการคูณเคอร์เนล

In [ ]:
PADDING_MODES = {"replicate": "edge", "zero": "constant", "reflect": "reflect"}


def filter2d(image, kernel, padding="replicate"):
    """กรองภาพแบบสหสัมพันธ์ตามรหัสเทียม FILTER2D (หัวข้อ 6.2.1) คืนภาพขนาดเท่าเดิมชนิด float"""
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    r = kernel.shape[0] // 2
    padded = np.pad(image, r, mode=PADDING_MODES[padding])
    height, width = image.shape
    out = np.zeros((height, width))
    for i in range(height):
        for j in range(width):
            neighborhood = padded[i:i + 2 * r + 1, j:j + 2 * r + 1]
            out[i, j] = np.sum(kernel * neighborhood)
    return out


def median_filter(image, size=3, padding="replicate"):
    """ตัวกรองมัธยฐาน (ไม่เชิงเส้น): เรียงค่าในหน้าต่างแล้วเลือกค่ากลาง"""
    image = np.asarray(image, dtype=float)
    r = size // 2
    padded = np.pad(image, r, mode=PADDING_MODES[padding])
    out = np.zeros(image.shape)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            out[i, j] = np.median(padded[i:i + size, j:j + size])
    return out


mean_own = filter2d(METAL, BOX_KERNEL)
gauss_own = filter2d(METAL, GAUSS_KERNEL)
median_own = median_filter(METAL)
display(pd.DataFrame({
    "g(1, 3) (was 255)": [mean_own[1, 3], gauss_own[1, 3], median_own[1, 3]],
    "g(1, 2) (was 100)": [mean_own[1, 2], gauss_own[1, 2], median_own[1, 2]],
}, index=["mean", "Gaussian", "median"]).round(1))
compare("ค่าเฉลี่ยที่ (1, 3)", mean_own[1, 3], 117.3, 0.05)
compare("ค่าเฉลี่ยที่ (1, 2)", mean_own[1, 2], 117.4, 0.05)
compare("เกาส์เซียนที่ (1, 3)", gauss_own[1, 3], 138.8, 0.05)
compare("มัธยฐานที่ (1, 3)", median_own[1, 3], 100, 1e-9)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (title, img) in zip(axes, [("original (Example 6.4)", METAL), ("mean 3x3", mean_own),
                                   ("Gaussian 3x3", gauss_own), ("median 3x3", median_own)]):
    show_matrix(img, title, ax, vmin=90, vmax=160)
plt.tight_layout()
plt.show()

### 2.2 เทียบกับฟังก์ชันของ OpenCV

ฟังก์ชันของ OpenCV ให้ผลเดียวกัน (ต่างกันเพียงการปัดเศษ) เมื่อเลือกวิธีเติมขอบให้ตรงกัน ข้อควรรู้คือ

- ค่าเริ่มต้นของการเติมขอบใน OpenCV คือ `BORDER_REFLECT_101` (สะท้อนแบบกระจกไม่ซ้ำพิกเซลขอบ) ไม่ใช่การทำซ้ำค่าขอบ ด้านล่างจึงระบุ `BORDER_REPLICATE` ให้ตรงกับ `filter2d`
- `cv2.GaussianBlur` ขนาด 3 × 3 ที่ `sigma = 0` ใช้เคอร์เนล $[1, 2, 1]/4$ ในแต่ละแกน ซึ่งคูณกันได้เคอร์เนล $\frac{1}{16}$ ในตารางที่ 6.2 พอดี
- `cv2.filter2D` คำนวณแบบ **สหสัมพันธ์** (ไม่พลิกเคอร์เนล) เหมือนชั้นคอนโวลูชันของ CNN ส่วน `scipy.ndimage.convolve` คำนวณคอนโวลูชันตามนิยามที่พลิกเคอร์เนล ผลต่างกันเมื่อเคอร์เนลไม่สมมาตร เช่น Sobel

In [ ]:
metal_f = METAL.astype(np.float32)
replicate = cv2.BORDER_REPLICATE
opencv_results = {
    "cv2.filter2D(box)": (cv2.filter2D(metal_f, -1, BOX_KERNEL.astype(np.float32), borderType=replicate), mean_own),
    "cv2.blur": (cv2.blur(metal_f, (3, 3), borderType=replicate), mean_own),
    "cv2.GaussianBlur": (cv2.GaussianBlur(metal_f, (3, 3), 0, borderType=replicate), gauss_own),
    "cv2.medianBlur": (cv2.medianBlur(METAL, 3).astype(float), median_own),
}
for name, (cv_img, own_img) in opencv_results.items():
    print(f"{name:20s} ต่างจากฟังก์ชันที่เขียนเองสูงสุด {np.abs(cv_img - own_img).max():.2e}")
print("\ncv2.getGaussianKernel(3, 0) =", cv2.getGaussianKernel(3, 0).ravel())
print("ถ้าใช้ภาพ uint8: cv2.blur ที่ (1, 3) =", cv2.blur(METAL, (3, 3))[1, 3], "(ปัดเป็นจำนวนเต็ม)")

# สหสัมพันธ์กับคอนโวลูชันบนแถวที่มีขอบมืด → สว่าง
step = np.tile([0, 0, 0, 100, 100, 100], (3, 1)).astype(float)
print("\nสหสัมพันธ์ (cv2.filter2D) ด้วย SOBEL_X :", cv2.filter2D(step, -1, SOBEL_X)[1])
print("คอนโวลูชัน (scipy.ndimage.convolve)    :", ndimage.convolve(step, SOBEL_X, mode="nearest")[1])

คอนโวลูชันตามนิยามให้เครื่องหมายกลับกัน เพราะพลิกเคอร์เนลก่อนคูณ สำหรับ CNN ความต่างนี้ไม่สำคัญ เพราะค่าในเคอร์เนลเรียนรู้จากข้อมูล (หัวข้อ 6.2.1)

### 2.3 ทดลองกับภาพผิวลูกถ้วยจำลอง

สร้างภาพผิวลูกถ้วยสีอ่อนขนาด 120 × 160 พิกเซลที่มีรอยร้าวกว้างราว 2 พิกเซล แล้วเติมสัญญาณรบกวนแบบจุด (Salt-and-Pepper) ร้อยละ 5 เปรียบเทียบตัวกรองด้วยตัวชี้วัดสองตัว

- **MAE** — ความคลาดเคลื่อนสัมบูรณ์เฉลี่ยเทียบกับภาพที่ไม่มีสัญญาณรบกวน (ยิ่งต่ำยิ่งดี)
- **ความเปรียบต่างของรอยร้าว** — ความสว่างเฉลี่ยของผิวลบความสว่างเฉลี่ยบนรอยร้าว (ยิ่งใกล้ภาพจริงยิ่งรักษารายละเอียดได้)

In [ ]:
CRACK_POINTS = np.array([[20, 5], [45, 40], [60, 62], [95, 80], [125, 112]], dtype=np.int32)  # จุด (x, y)


def make_insulator_surface(seed=RANDOM_SEED, height=120, width=160):
    """ภาพผิวลูกถ้วยจำลอง (แสงไม่สม่ำเสมอเล็กน้อย) ที่มีรอยร้าว คืน (ภาพ uint8, หน้ากากรอยร้าว)"""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:height, 0:width]
    surface = 175 + 15 * xx / width - 10 * yy / height + rng.normal(0, 3, (height, width))
    surface = np.clip(surface, 0, 255).astype(np.uint8)
    cv2.polylines(surface, [CRACK_POINTS], False, 70, 2)
    crack = np.zeros((height, width), dtype=np.uint8)
    cv2.polylines(crack, [CRACK_POINTS], False, 1, 2)
    return surface, crack.astype(bool)


def add_salt_pepper(image, fraction, seed=RANDOM_SEED):
    """สุ่มพิกเซลให้เป็น 0 หรือ 255 ตามสัดส่วนที่กำหนด"""
    rng = np.random.default_rng(seed)
    noisy = image.copy()
    hit = rng.random(image.shape) < fraction
    noisy[hit] = rng.choice(np.array([0, 255], dtype=np.uint8), size=hit.sum())
    return noisy


def crack_contrast(image, crack):
    return image[~crack].mean() - image[crack].mean()


surface, surface_crack = make_insulator_surface()
noisy_surface = add_salt_pepper(surface, 0.05)
filtered = {
    "noisy (no filter)": noisy_surface,
    "mean 3x3": cv2.blur(noisy_surface, (3, 3)),
    "Gaussian 3x3": cv2.GaussianBlur(noisy_surface, (3, 3), 0),
    "median 3x3": cv2.medianBlur(noisy_surface, 3),
    "mean 7x7": cv2.blur(noisy_surface, (7, 7)),
    "median 7x7": cv2.medianBlur(noisy_surface, 7),
}
rows = [{"filter": name, "MAE vs clean": np.abs(img.astype(float) - surface).mean(),
         "crack contrast": crack_contrast(img.astype(float), surface_crack)} for name, img in filtered.items()]
print(f"ความเปรียบต่างของรอยร้าวในภาพจริง (ไม่มีสัญญาณรบกวน): {crack_contrast(surface.astype(float), surface_crack):.1f}")
display(pd.DataFrame(rows).set_index("filter").round(1))

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, img) in zip(axes.ravel(), filtered.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

ผลสอดคล้องกับตัวอย่างที่ 6.4 คือตัวกรองค่าเฉลี่ยและเกาส์เซียนกระจายจุดรบกวนออกเป็นหย่อมจาง ๆ ส่วนตัวกรองมัธยฐาน 3 × 3 กำจัดจุดรบกวนได้เกือบหมดและยังรักษารอยร้าวไว้ แต่เมื่อขยายเป็น 7 × 7 รอยร้าวกว้าง 2 พิกเซลจางลง (ความเปรียบต่างลดลง) ซึ่งคือ "ราคาของการลดสัญญาณรบกวน" ในหัวข้อ 6.2.1

**คำถามชวนคิด** ถ้าสัญญาณรบกวนเป็นแบบเกาส์เซียน (ทุกพิกเซลแกว่งเล็กน้อย) แทนแบบจุด ผลการจัดอันดับตัวกรองจะเปลี่ยนหรือไม่ ลองแก้เซลล์ด้านบนให้ใช้ `surface + rng.normal(0, 15, ...)` แล้วสังเกต

## ส่วนที่ 3 การหาขอบภาพ

(หนังสือหัวข้อ 6.2.2)

### 3.1 Sobel บนรอยร้าวแนวตั้ง (ตัวอย่างที่ 6.5)

In [ ]:
vertical_crack = np.tile([200, 200, 60, 200, 200], (5, 1)).astype(float)
gx_own = filter2d(vertical_crack, SOBEL_X)
gy_own = filter2d(vertical_crack, SOBEL_Y)
gx_cv = cv2.Sobel(vertical_crack, cv2.CV_64F, 1, 0, ksize=3)

print("ความสว่างแถว 2      :", vertical_crack[2])
print("Gx (filter2d) แถว 2 :", gx_own[2])
print("Gx (cv2.Sobel) แถว 2:", gx_cv[2])
print("Gy แถว 2            :", gy_own[2])
print("ขนาดเกรเดียนต์ที่ c = 1, 2, 3:", np.hypot(gx_own, gy_own)[2, 1:4])
compare("สัดส่วนของค่าสูงสุดที่เป็นไปได้ 560/1020 (%)", 560 / 1020 * 100, 55, 0.5)

รอยร้าวเส้นเดียวให้ขอบสองเส้นขนาบข้าง ($G_x = -560$ และ $+560$) และกลางรอยร้าวมีเกรเดียนต์เป็นศูนย์ ตรงกับตัวอย่างที่ 6.5 (ค่าที่ $c = 0$ และ $c = 4$ ขึ้นกับวิธีเติมขอบ ซึ่งหนังสือไม่ได้คำนวณ)

### 3.2 ขนาดและทิศทางของเกรเดียนต์บนภาพจำลอง

In [ ]:
smoothed = cv2.GaussianBlur(surface, (3, 3), 0).astype(np.float64)
gx = cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.hypot(gx, gy)
direction = np.degrees(np.arctan2(gy, gx))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].imshow(surface, cmap="gray")
axes[0].set_title("simulated insulator surface")
im = axes[1].imshow(magnitude, cmap="magma")
axes[1].set_title("gradient magnitude |grad f|")
plt.colorbar(im, ax=axes[1], fraction=0.046)
strong = magnitude > 200
im = axes[2].imshow(np.where(strong, direction, np.nan), cmap="twilight", vmin=-180, vmax=180)
axes[2].set_title("direction (deg) where |grad f| > 200")
plt.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"สองฝั่งของรอยร้าวมีทิศทางเกรเดียนต์ต่างกันราว 180° (ชี้ออกจากรอยร้าวไปทางที่สว่างกว่า) | ขนาดสูงสุด {magnitude.max():.0f}")

### 3.3 ตัวตรวจหาขอบของ Canny: ผลของค่าเกณฑ์

ใช้ภาพผิวลูกถ้วยที่เติมสัญญาณรบกวนแบบเกาส์เซียน แล้วลองค่าเกณฑ์สามคู่ (`cv2.Canny(image, T_low, T_high)`) สังเกตว่าเกณฑ์ต่ำได้จุดรบกวนมาก เกณฑ์สูงทำให้รอยร้าวขาด

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
grainy = np.clip(surface + rng.normal(0, 10, surface.shape), 0, 255).astype(np.uint8)
blurred = cv2.GaussianBlur(grainy, (5, 5), 1.2)        # ขั้นที่ 1 ของ Canny: ลดสัญญาณรบกวน

settings = [(10, 30), (40, 120), (150, 400)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
axes[0].imshow(grainy, cmap="gray")
axes[0].set_title("noisy surface")
for ax, (low, high) in zip(axes[1:], settings):
    edges = cv2.Canny(blurred, low, high)
    on_crack = (edges > 0) & cv2.dilate(surface_crack.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    ax.imshow(edges, cmap="gray")
    ax.set_title(f"T_low={low}, T_high={high}\n{(edges > 0).sum()} edge px, {on_crack.sum()} near crack")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.4 ไล่ขั้นฮิสเทอรีซิส (ตัวอย่างที่ 6.6)

ฟังก์ชัน `hysteresis` รับขนาดเกรเดียนต์ของพิกเซลและรายการเพื่อนบ้าน แล้วทำขั้นที่ 4–5 ของ Canny: แบ่งเป็นขอบแข็ง/อ่อน แล้วเพิ่มขอบอ่อนที่ติดกับขอบใน $E$ ทีละรอบ

In [ ]:
def hysteresis(magnitudes, neighbors, t_low, t_high):
    """ค่าเกณฑ์สองระดับ + ติดตามขอบ คืน (ระดับของแต่ละพิกเซล, รายการพิกเซลที่เพิ่มในแต่ละรอบ, E)"""
    level = {p: "strong" if m >= t_high else "weak" if m >= t_low else "suppressed" for p, m in magnitudes.items()}
    edges = {p for p, lv in level.items() if lv == "strong"}
    rounds = []
    while True:
        added = {q for p in edges for q in neighbors[p] if level[q] == "weak" and q not in edges}
        if not added:
            break
        rounds.append(sorted(added))
        edges |= added
    return level, rounds, sorted(edges)


EX66_MAGNITUDES = {"P1": 350, "P2": 250, "P3": 180, "P4": 90, "P5": 120, "P6": 320, "Q": 150}
EX66_NEIGHBORS = {"P1": ["P2"], "P2": ["P1", "P3"], "P3": ["P2", "P4"], "P4": ["P3", "P5"],
                  "P5": ["P4", "P6"], "P6": ["P5"], "Q": []}
level, rounds, final_edges = hysteresis(EX66_MAGNITUDES, EX66_NEIGHBORS, t_low=100, t_high=300)
display(pd.DataFrame({"magnitude": EX66_MAGNITUDES, "step 4": level,
                      "step 5": {p: "edge" if p in final_edges else "-" for p in EX66_MAGNITUDES}}).T)
print("รอบที่เพิ่มขอบอ่อน:", rounds)
print("E =", final_edges, "(หนังสือ: P1, P2, P3, P5, P6)")
print("เกณฑ์เดียวที่ 300:", [p for p, m in EX66_MAGNITUDES.items() if m >= 300],
      "| เกณฑ์เดียวที่ 100:", [p for p, m in EX66_MAGNITUDES.items() if m >= 100])

## ส่วนที่ 4 Thresholding, Morphology และองค์ประกอบเชื่อมต่อ

(หนังสือหัวข้อ 6.2.3)

### 4.1 วิธีของ Otsu บนฮิสโทแกรม 8 ระดับ (ตัวอย่างที่ 6.7)

`otsu_table` ไล่ทุกค่าเกณฑ์ $t$ ตามรหัสเทียม OTSU แล้วคำนวณ $\sigma_B^2 = \omega_0\omega_1(\mu_0 - \mu_1)^2$ (กลุ่มมืดคือค่า $\le t$)

In [ ]:
def otsu_table(hist):
    """ตารางไล่ค่าเกณฑ์ของ Otsu จากฮิสโทแกรม (ข้ามค่าเกณฑ์ที่กลุ่มใดกลุ่มหนึ่งว่าง)"""
    hist = np.asarray(hist, dtype=float)
    levels = np.arange(len(hist))
    total = hist.sum()
    rows = []
    for t in range(len(hist) - 1):
        n0, n1 = hist[:t + 1].sum(), hist[t + 1:].sum()
        if n0 == 0 or n1 == 0:
            continue
        w0, w1 = n0 / total, n1 / total
        mu0 = (levels[:t + 1] * hist[:t + 1]).sum() / n0
        mu1 = (levels[t + 1:] * hist[t + 1:]).sum() / n1
        rows.append({"t": t, "N0": int(n0), "N1": int(n1), "w0": w0, "w1": w1, "mu0": mu0, "mu1": mu1,
                     "(mu0-mu1)^2": (mu0 - mu1) ** 2, "sigma_B^2": w0 * w1 * (mu0 - mu1) ** 2})
    return pd.DataFrame(rows).set_index("t")


EX67_HIST = [0, 3, 5, 2, 0, 1, 6, 3]
otsu_df = otsu_table(EX67_HIST)
display(otsu_df.round(3))
best_t = otsu_df["sigma_B^2"].idxmax()          # idxmax คืนค่าแรกเมื่อเสมอ (t = 3 ก่อน t = 4)
print("t* =", best_t)
compare("sigma_B^2 ที่ t*", otsu_df.loc[best_t, "sigma_B^2"], 4.623, 5e-4)

meter_pixels = np.repeat(np.arange(8), EX67_HIST).astype(np.uint8).reshape(1, -1)   # 20 พิกเซลตามฮิสโทแกรม
t_cv, _ = cv2.threshold(meter_pixels, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("cv2.threshold(..., THRESH_OTSU) ได้ค่าเกณฑ์", t_cv)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(8), EX67_HIST, color=COLORS["blue"])
ax.axvline(best_t + 0.5, color=COLORS["orange"], ls="--", label=f"Otsu: dark <= {best_t}")
ax.set_xlabel("gray level k")
ax.set_ylabel("count n_k")
ax.legend()
plt.show()

### 4.2 ภาพหน้าจอมิเตอร์แบบเจ็ดส่วน

สร้างภาพจำลองของหน้าจอมิเตอร์ที่แสดงค่าแรงดัน "230" เป็นตัวเลขสว่างบนพื้นมืด มีแสงจากด้านซ้ายทำให้พื้นหลังไม่สม่ำเสมอ มีจุดฝุ่นสว่างกระจาย และขีดกลาง (ส่วน g) ของเลข 2 ขาดเป็นช่องกว้าง 2 พิกเซลเพราะหน้าจอเสื่อม แล้วทำตามขั้นตอน: กรองเกาส์เซียน → ค่าเกณฑ์ Otsu → การเปิด (ลบฝุ่น) → การปิด (เชื่อมรอยขาด) → นับองค์ประกอบเชื่อมต่อ

In [ ]:
# ส่วน a–g ของตัวเลขเจ็ดส่วน: (แถวเริ่ม, แถวสิ้นสุด, คอลัมน์เริ่ม, คอลัมน์สิ้นสุด) ในกรอบตัวเลขสูง 36 กว้าง 20
SEGMENT_BOXES = {"a": (0, 4, 2, 18), "b": (2, 18, 16, 20), "c": (18, 34, 16, 20), "d": (32, 36, 2, 18),
                 "e": (18, 34, 0, 4), "f": (2, 18, 0, 4), "g": (16, 20, 2, 18)}
DIGIT_SEGMENTS = {"0": "abcdef", "1": "bc", "2": "abged", "3": "abgcd", "4": "fgbc",
                  "5": "afgcd", "6": "afgedc", "7": "abc", "8": "abcdefg", "9": "abcdfg"}


def make_meter_image(text="230", seed=RANDOM_SEED, n_dust=25, broken=("2", "g")):
    """ภาพระดับเทาจำลองของหน้าจอเจ็ดส่วน คืน (ภาพ uint8, จำนวนตัวเลข)"""
    rng = np.random.default_rng(seed)
    height, width = 56, 16 + 28 * len(text)
    xx = np.arange(width)[None, :]
    image = 35 + 45 * (1 - xx / width) + rng.normal(0, 6, (height, width))   # แสงจากหน้าต่างด้านซ้าย
    for k, digit in enumerate(text):
        top, left = 10, 10 + 28 * k
        for seg in DIGIT_SEGMENTS[digit]:
            r0, r1, c0, c1 = SEGMENT_BOXES[seg]
            image[top + r0:top + r1, left + c0:left + c1] = rng.normal(200, 8, (r1 - r0, c1 - c0))
            if (digit, seg) == broken:
                mid = left + (c0 + c1) // 2
                image[top + r0:top + r1, mid - 1:mid + 1] = 60      # รอยขาดกว้าง 2 พิกเซล
    for _ in range(n_dust):                                          # จุดฝุ่นสะท้อนแสง
        r, c = int(rng.integers(0, height)), int(rng.integers(0, width))
        image[r, c] = rng.uniform(170, 230)
    return np.clip(image, 0, 255).astype(np.uint8), len(text)


meter, n_digits = make_meter_image()
meter_blur = cv2.GaussianBlur(meter, (3, 3), 0)
otsu_t, meter_binary = cv2.threshold(meter_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel3 = np.ones((3, 3), np.uint8)
meter_open = cv2.morphologyEx(meter_binary, cv2.MORPH_OPEN, kernel3)
meter_clean = cv2.morphologyEx(meter_open, cv2.MORPH_CLOSE, kernel3)

steps = [("raw image", meter), (f"Otsu (t = {otsu_t:.0f})", meter_binary), ("opening", meter_open), ("opening + closing", meter_clean)]
fig, axes = plt.subplots(1, 4, figsize=(15, 2.8))
for ax, (title, img) in zip(axes, steps):
    if img is not meter:
        n_comp = cv2.connectedComponents(img, connectivity=8)[0] - 1   # ลบป้ายของพื้นหลัง
        title += f"\n{n_comp} components"
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

n_final, labels, stats, centroids = cv2.connectedComponentsWithStats(meter_clean, connectivity=8)
display(pd.DataFrame(stats[1:], columns=["x", "y", "width", "height", "area (px)"]).rename_axis("component"))
print(f"จำนวนตัวเลขจริง {n_digits} | องค์ประกอบหลังทำความสะอาด {n_final - 1}")

หลังค่าเกณฑ์ Otsu ภาพสองค่ามีองค์ประกอบมากเกินจริง เพราะจุดฝุ่นแต่ละจุดนับเป็นวัตถุ และเลข 2 ขาดเป็นสองชิ้น การเปิดลบจุดฝุ่นที่เล็กกว่าส่วนประกอบโครงสร้าง 3 × 3 และการปิดเชื่อมรอยขาดที่แคบกว่า 3 พิกเซล จนเหลือองค์ประกอบเท่าจำนวนตัวเลขพอดี ซึ่งพร้อมส่งต่อให้ขั้นอ่านตัวเลข (เช่น ตรวจว่าส่วน a–g ใดสว่าง)

### 4.3 การเปิดและการปิดบนภาพในตัวอย่างที่ 6.8

ใช้ `cv2.morphologyEx` กับภาพสองค่าในหนังสือโดยตรง (แบบฝึกปฏิบัติข้อ 2 จะให้เขียนการกร่อนและการพองตัวเอง)

In [ ]:
case_a = np.zeros((5, 7), np.uint8)
case_a[1:4, 1:4] = 1          # วัตถุ 3 × 3
case_a[2, 5] = 1              # จุดฝุ่นที่ (2, 5)
case_b = np.zeros((6, 9), np.uint8)
case_b[2:4, 2:4] = 1          # ขีดซ้าย
case_b[2:4, 5:7] = 1          # ขีดขวา (ขาดที่คอลัมน์ 4)

print_binary(case_a, "กรณี ก ภาพเดิม")
print_binary(cv2.erode(case_a, kernel3), "หลังกร่อน")
opened_a = cv2.morphologyEx(case_a, cv2.MORPH_OPEN, kernel3)
print_binary(opened_a, f"หลังเปิด ({case_a.sum()} → {opened_a.sum()} พิกเซล)")
print()
print_binary(case_b, "กรณี ข ภาพเดิม")
print_binary(cv2.dilate(case_b, kernel3), "หลังพองตัว")
closed_b = cv2.morphologyEx(case_b, cv2.MORPH_CLOSE, kernel3)
print_binary(closed_b, f"หลังปิด ({closed_b.sum()} พิกเซล)")

### 4.4 องค์ประกอบเชื่อมต่อและขนาดจริงของรอยร้าว (ตัวอย่างที่ 6.9)

นับองค์ประกอบของพิกเซลรอยร้าวในตัวอย่างที่ 6.1 ทั้งแบบ 4 ทิศและ 8 ทิศ เทียบกับรอยร้าวที่เฉียงทีละหนึ่งพิกเซลพอดี แล้วแปลงเป็นความยาวและพื้นที่จริงที่ GSD 0.833 mm ต่อพิกเซล

In [ ]:
crack_u8 = crack_mask.astype(np.uint8)
diagonal = np.eye(3, dtype=np.uint8)
for name, img in [("รอยร้าวตัวอย่างที่ 6.1", crack_u8), ("เส้นทแยง (0,0),(1,1),(2,2)", diagonal)]:
    n4 = cv2.connectedComponents(img, connectivity=4)[0] - 1
    n8 = cv2.connectedComponents(img, connectivity=8)[0] - 1
    print(f"{name:28s}: 4 ทิศ = {n4} องค์ประกอบ | 8 ทิศ = {n8} องค์ประกอบ")

gsd_10m = gsd_mm(10_000, **DRONE_CAMERA)
crack_pixels = np.argwhere(crack_mask)
top_end, bottom_end = crack_pixels[0], crack_pixels[-1]            # (0, 3) และ (5, 1)
length_px = np.linalg.norm(bottom_end - top_end) + 1                 # รวมครึ่งพิกเซลที่ปลายทั้งสอง
print(f"\nปลายรอยร้าว {tuple(int(v) for v in top_end)} → {tuple(int(v) for v in bottom_end)}")
compare("ความยาว (พิกเซล)", length_px, 6.4, 0.05)
compare("ความยาว (mm)", length_px * gsd_10m, 5.3, 0.05)
compare("พื้นที่ (mm²)", crack_mask.sum() * gsd_10m ** 2, 5.6, 0.05)

## ส่วนที่ 5 การจำแนกภาพ: จากคุณลักษณะที่ออกแบบเองสู่ส่วนหลักที่แช่แข็ง

(หนังสือหัวข้อ 6.3)

### 5.1 ชุดข้อมูลภาพผิวลูกถ้วยจำลอง

ชุดข้อมูลสาธารณะจริง เช่น ELPV (ภาพ EL ของเซลล์แสงอาทิตย์) มีขนาดหลายร้อย MB เพื่อให้รันได้ทันทีบน CPU จึงสร้างภาพผิวลูกถ้วยขนาด 32 × 32 พิกเซลสามคลาส คลาสละ 200 ภาพ

- **ปกติ (normal)** — ผิวเคลือบที่มีลายคลื่นเบา ๆ สัญญาณรบกวน และแสงเอียงต่างกันทุกภาพ
- **สกปรก (dirty)** — มีคราบเป็นจุดหรือวงรีมืด 2–6 จุด (เช่น มูลนก)
- **แตก (cracked)** — มีรอยร้าวเส้นตรงกว้าง 1 พิกเซล ความยาวและความเปรียบต่างต่างกัน บางภาพจางมาก

In [ ]:
CLASSES = ["normal", "dirty", "cracked"]


def make_patch(kind, rng, size=32):
    """ภาพผิวลูกถ้วยจำลองหนึ่งภาพตามคลาสที่กำหนด"""
    base = rng.uniform(140, 210)
    yy, xx = np.mgrid[0:size, 0:size]
    image = base + rng.uniform(-0.8, 0.8) * (xx - size / 2) + rng.uniform(-0.8, 0.8) * (yy - size / 2)
    image = image + rng.uniform(0, 8) * np.sin(xx * rng.uniform(0.3, 1.2) + yy * rng.uniform(0.3, 1.2))
    image = (image + rng.normal(0, rng.uniform(2, 7), (size, size))).astype(np.float32)
    if kind == "dirty":
        for _ in range(rng.integers(2, 7)):
            center = (int(rng.integers(3, size - 3)), int(rng.integers(3, size - 3)))
            axes = (int(rng.integers(1, 4)), int(rng.integers(1, 4)))
            cv2.ellipse(image, center, axes, float(rng.uniform(0, 180)), 0, 360, float(base - rng.uniform(25, 80)), -1)
    elif kind == "cracked":
        angle = rng.uniform(0, np.pi)
        cx, cy = rng.uniform(10, size - 10, size=2)
        half_length = rng.uniform(5, 15)
        p1 = (int(cx - half_length * np.cos(angle)), int(cy - half_length * np.sin(angle)))
        p2 = (int(cx + half_length * np.cos(angle)), int(cy + half_length * np.sin(angle)))
        cv2.line(image, p1, p2, float(base - rng.uniform(20, 100)), 1)
    return np.clip(image, 0, 255).astype(np.uint8)


def make_patch_dataset(n_per_class=200, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for label, kind in enumerate(CLASSES):
        for _ in range(n_per_class):
            images.append(make_patch(kind, rng))
            labels.append(label)
    return np.array(images), np.array(labels)


patches, patch_labels = make_patch_dataset()
train_idx, test_idx = train_test_split(np.arange(len(patches)), test_size=0.3, stratify=patch_labels,
                                       random_state=RANDOM_SEED)
print("ภาพทั้งหมด", patches.shape, "| ชุดฝึก", len(train_idx), "| ชุดทดสอบ", len(test_idx))

fig, axes = plt.subplots(3, 8, figsize=(12, 4.8))
for row, label in enumerate(range(3)):
    examples = np.flatnonzero(patch_labels == label)[:8]
    for ax, i in zip(axes[row], examples):
        ax.imshow(patches[i], cmap="gray", vmin=0, vmax=255)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[row, 0].set_ylabel(CLASSES[label])
plt.suptitle("simulated insulator patches (32 x 32)")
plt.tight_layout()
plt.show()

> **หมายเหตุ** ภาพแต่ละภาพในชุดนี้สร้างแยกกันอย่างอิสระ จึงแบ่งชุดฝึก/ทดสอบแบบสุ่มรายภาพได้ ในงานจริงที่ถ่ายลูกถ้วยใบเดียวกันหลายภาพ ต้องแบ่ง **ตามวัตถุจริง** (หัวข้อ 6.3.2) เช่นใช้ `GroupShuffleSplit` ของ scikit-learn

### 5.2 คุณลักษณะที่ออกแบบเองและจุดที่ล้มเหลว (ตัวอย่างที่ 6.10)

ทำแบบตัวอย่างที่ 6.10: คำนวณ **ความหนาแน่นของขอบ** (ร้อยละของพิกเซลที่ `cv2.Canny` ระบุว่าเป็นขอบ) แล้วเรียนรู้ค่าเกณฑ์ "ขอบหนาแน่นเกินเกณฑ์ = แตก" จากภาพฝึกคลาสปกติกับคลาสแตกเท่านั้น (เหมือนวิศวกรที่ยังไม่เคยเห็นภาพคราบสกปรก) จากนั้นนำกฎไปใช้กับชุดทดสอบทั้งสามคลาส

In [ ]:
def handcrafted_features(image):
    """คุณลักษณะสองตัวแบบตัวอย่างที่ 6.10: ความหนาแน่นของขอบ (%) และสัดส่วนพิกเซลมืด (%)"""
    edges = cv2.Canny(image, 50, 150)
    dark = image < (np.median(image) - 25)
    return [np.count_nonzero(edges) / edges.size * 100, dark.mean() * 100]


hand_feats = np.array([handcrafted_features(img) for img in patches])
edge_density = hand_feats[:, 0]

# เรียนรู้ค่าเกณฑ์จากภาพฝึกคลาส normal (0) กับ cracked (2) โดยเลือกจุดกึ่งกลางที่ถูกมากที่สุด
learn = train_idx[np.isin(patch_labels[train_idx], [0, 2])]
values = np.unique(edge_density[learn])
candidates = (values[:-1] + values[1:]) / 2
accuracy = [np.mean((edge_density[learn] > t) == (patch_labels[learn] == 2)) for t in candidates]
edge_threshold = candidates[int(np.argmax(accuracy))]
print(f"กฎที่เรียนรู้ได้: ถ้าความหนาแน่นของขอบ > {edge_threshold:.2f}% แล้ว 'แตก' (ความถูกต้องบนภาพฝึก normal/cracked = {max(accuracy):.3f})")

flag_rate = {CLASSES[k]: np.mean(edge_density[test_idx][patch_labels[test_idx] == k] > edge_threshold) for k in range(3)}
display(pd.Series(flag_rate, name="fraction flagged as 'cracked' (test set)").round(3).to_frame())

fig, ax = plt.subplots(figsize=(7, 3.2))
for k, color in zip(range(3), [COLORS["green"], COLORS["gray"], COLORS["orange"]]):
    ax.hist(edge_density[patch_labels == k], bins=np.arange(0, 16, 0.5), alpha=0.6, color=color, label=CLASSES[k])
ax.axvline(edge_threshold, color="black", ls="--", label="learned threshold")
ax.set_xlabel("edge density (% of pixels)")
ax.set_ylabel("number of patches")
ax.legend()
plt.show()

กฎแยกภาพปกติกับภาพแตกได้ดี แต่เกือบทุกภาพคราบสกปรกถูกทำนายว่า "แตก" (FP แบบภาพ N1 ในตัวอย่างที่ 6.10) และรอยร้าวที่จางมากถูกพลาด (FN แบบภาพ N2) เพราะความหนาแน่นของขอบวัดว่า "มีเส้นและจุดมากเพียงใด" ไม่ได้วัดว่า "มีรอยแตกหรือไม่"

### 5.3 นับพารามิเตอร์: ชั้นเชื่อมต่อเต็ม ชั้นคอนโวลูชัน และ ResNet-18 (ตัวอย่างที่ 6.12)

ฟังก์ชัน `resnet18_parameters` นับพารามิเตอร์ของ ResNet-18 จากสถาปัตยกรรมโดยตรง (ชั้นคอนโวลูชันไม่มีไบแอส แต่ละ Batch Normalization มีพารามิเตอร์ที่ฝึกได้ 2 ชุด) โดยไม่ต้องติดตั้งไลบรารีการเรียนรู้เชิงลึก

In [ ]:
def conv_parameters(kernel_size, in_channels, out_channels, bias=True):
    return kernel_size * kernel_size * in_channels * out_channels + (out_channels if bias else 0)


def resnet18_parameters(n_classes=1000):
    """จำนวนพารามิเตอร์ของ ResNet-18 คืน (ส่วนหลัก, ส่วนหัว)"""
    def bn(channels):
        return 2 * channels                              # gamma และ beta
    backbone = conv_parameters(7, 3, 64, bias=False) + bn(64)
    in_ch = 64
    for out_ch in (64, 128, 256, 512):                   # 4 ระยะ ระยะละ 2 บล็อก
        for block in range(2):
            backbone += conv_parameters(3, in_ch, out_ch, bias=False) + bn(out_ch)
            backbone += conv_parameters(3, out_ch, out_ch, bias=False) + bn(out_ch)
            if block == 0 and out_ch != 64:              # ทางลัดแบบ 1 × 1 เมื่อขนาดเปลี่ยน
                backbone += conv_parameters(1, in_ch, out_ch, bias=False) + bn(out_ch)
            in_ch = out_ch
    head = 512 * n_classes + n_classes
    return backbone, head


fc_weights = 224 * 224 * 3 * 1000
conv_params = conv_parameters(3, 3, 64)
print(f"ชั้นเชื่อมต่อเต็ม 1,000 หน่วยบนภาพ 224×224×3: {fc_weights:,} ค่าน้ำหนัก")
print(f"ชั้นคอนโวลูชัน 3×3 จำนวน 64 ตัว: {conv_params:,} พารามิเตอร์ (น้อยกว่า {fc_weights / conv_params:,.0f} เท่า)")

backbone, imagenet_head = resnet18_parameters(1000)
_, new_head = resnet18_parameters(3)
print(f"\nResNet-18: รวม {backbone + imagenet_head:,} | ชั้นจำแนกเดิม {imagenet_head:,} | ส่วนหลัก {backbone:,}")
print(f"ส่วนหัวใหม่ 3 คลาส: {new_head:,} พารามิเตอร์ = {new_head / backbone * 100:.3f}% ของส่วนหลัก")
compare("ส่วนหัวใหม่", new_head, 1539, 0)
compare("ส่วนหลัก (ล้าน)", backbone / 1e6, 11.2, 0.05)

### 5.4 "ส่วนหลักที่แช่แข็ง + ส่วนหัวที่ฝึก" ฉบับย่อส่วน

แนวคิดของการสกัดคุณลักษณะ (Feature Extraction) ในหัวข้อ 6.3.2 คือใช้ส่วนหลักที่ **ไม่ฝึกแล้ว** แปลงภาพเป็นเวกเตอร์คุณลักษณะ แล้วฝึกเฉพาะส่วนหัวซึ่งเทียบเท่า Logistic Regression หลายคลาส (ตีความของตัวอย่างที่ 6.12) ส่วนนี้จำลองแนวคิดเดียวกันบน CPU โดยไม่ต้องดาวน์โหลดโมเดล

- **ส่วนหลักที่แช่แข็ง** = ชั้นคอนโวลูชันหนึ่งชั้นที่มีเคอร์เนล 7 ตัวซึ่งออกแบบไว้แล้วและไม่ถูกปรับ (ตัวหาเส้น 4 ทิศ, Sobel 2 ทิศ, ตัวหาจุดแบบ center–surround) ตามด้วย ReLU ทั้งสองขั้ว (ตอบสนองต่อสิ่งที่มืดกว่าและสว่างกว่า) และการรวมค่า (ค่าสูงสุด ค่าเฉลี่ย และการรวมค่าเฉลี่ยแบบ 4 × 4)
- **ส่วนหัว** = `LogisticRegression` ที่ฝึกบนเวกเตอร์คุณลักษณะ 56 ค่า

ในโมเดลจริงอย่าง ResNet-18 เคอร์เนลของส่วนหลัก **เรียนรู้มาจาก ImageNet** และมีหลายสิบชั้น แต่ขั้นตอนการใช้งานเหมือนกัน ส่วนที่ 5.6 (ทางเลือก) ทำแบบนั้นจริง

In [ ]:
def line_kernel(theta_deg, size=5):
    """เคอร์เนลหาเส้นตรงบางในทิศ theta (ผลรวมเป็นศูนย์ จึงไม่ตอบสนองต่อพื้นที่เรียบ)"""
    kernel = np.zeros((size, size), np.float32)
    c = size // 2
    theta = np.radians(theta_deg)
    for t in np.linspace(-c, c, 41):
        kernel[int(round(c + t * np.sin(theta))), int(round(c + t * np.cos(theta)))] = 1
    return kernel / kernel.sum() - 1 / size ** 2


gaussian_5 = cv2.getGaussianKernel(5, 1) @ cv2.getGaussianKernel(5, 1).T
FROZEN_KERNELS = {f"line {a} deg": line_kernel(a) for a in (0, 45, 90, 135)}
FROZEN_KERNELS["Sobel x"] = (SOBEL_X / 4).astype(np.float32)
FROZEN_KERNELS["Sobel y"] = (SOBEL_Y / 4).astype(np.float32)
FROZEN_KERNELS["spot"] = (gaussian_5 - 1 / 25).astype(np.float32)


def frozen_backbone(image):
    """คอนโวลูชัน → ReLU (สองขั้ว) → การรวมกลุ่ม คืนเวกเตอร์คุณลักษณะ 7 × 2 × 4 = 56 ค่า"""
    x = image.astype(np.float32) / 255.0
    features = []
    for kernel in FROZEN_KERNELS.values():
        response = cv2.filter2D(x, -1, kernel, borderType=cv2.BORDER_REFLECT)
        for activation in (np.maximum(-response, 0), np.maximum(response, 0)):
            pooled = cv2.resize(activation, (4, 4), interpolation=cv2.INTER_AREA)   # average pooling 8 × 8
            features += [activation.max(), activation.mean(), pooled.max(), pooled.std()]
    return np.array(features)


fig, axes = plt.subplots(1, len(FROZEN_KERNELS), figsize=(13, 2))
for ax, (name, kernel) in zip(axes, FROZEN_KERNELS.items()):
    ax.imshow(kernel, cmap="RdBu_r")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("frozen 'backbone' kernels (not trained)")
plt.show()

backbone_feats = np.array([frozen_backbone(img) for img in patches])
print("เวกเตอร์คุณลักษณะจากส่วนหลัก:", backbone_feats.shape)

In [ ]:
def fit_and_report(name, features, n_params):
    """ฝึกส่วนหัว Logistic Regression แล้วคืนตัววัดบนชุดทดสอบ"""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    model.fit(features[train_idx], patch_labels[train_idx])
    pred = model.predict(features[test_idx])
    truth = patch_labels[test_idx]
    return model, pred, {"model": name, "trainable params": n_params,
                         "accuracy": np.mean(pred == truth),
                         "macro-F1": f1_score(truth, pred, average="macro"),
                         "recall (cracked)": np.mean(pred[truth == 2] == 2)}


raw_pixels = patches.reshape(len(patches), -1) / 255.0
experiments = [
    ("raw pixels + logistic head", raw_pixels),
    ("2 hand-crafted features + logistic head", hand_feats),
    ("frozen backbone (56 features) + logistic head", backbone_feats),
]
results, predictions, models = [], {}, {}
for name, feats in experiments:
    n_params = feats.shape[1] * len(CLASSES) + len(CLASSES)        # ค่าน้ำหนัก + ไบแอสของส่วนหัว
    model, pred, row = fit_and_report(name, feats, n_params)
    results.append(row)
    predictions[name], models[name] = pred, model
display(pd.DataFrame(results).set_index("model").round(3))

best_name = experiments[-1][0]
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ConfusionMatrixDisplay(confusion_matrix(patch_labels[test_idx], predictions[best_name]),
                       display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("frozen backbone + head (test set)")
plt.show()
print(classification_report(patch_labels[test_idx], predictions[best_name], target_names=CLASSES, digits=3))

ข้อสังเกต

- **พิกเซลดิบ** มีพารามิเตอร์มากที่สุดแต่ได้ผลแย่ที่สุด เพราะรอยร้าวปรากฏได้ทุกตำแหน่งและทุกทิศ โมเดลเชิงเส้นบนพิกเซลต้องเรียนรู้ทุกตำแหน่งแยกกัน (ขาดการใช้ค่าน้ำหนักร่วมกันและการสมมูลภายใต้การเลื่อน)
- **คุณลักษณะที่ออกแบบเองสองตัว** ทำได้ค่อนข้างดีบนภาพจำลองที่ควบคุมได้ แต่ขึ้นกับว่าผู้ออกแบบคิดคุณลักษณะที่ถูกต้องไว้ล่วงหน้า (ข้อ 5.2 แสดงแล้วว่าคุณลักษณะเดียวล้มเหลวอย่างไร)
- **ส่วนหลักที่แช่แข็ง + ส่วนหัว** ได้ผลดีที่สุดด้วยพารามิเตอร์ที่ฝึกเพียงหลักร้อย เพราะเคอร์เนลคอนโวลูชันและการรวมค่าให้คุณลักษณะที่ไม่ขึ้นกับตำแหน่ง ซึ่งเป็นเหตุผลเดียวกับที่ Transfer Learning ได้ผลกับข้อมูลน้อย

### 5.5 ซอฟต์แมกซ์และตัววัดรายคลาส (ตัวอย่างที่ 6.11 และ 6.13)

ส่วนหัวของ `LogisticRegression` หลายคลาสให้ค่าลอจิตด้วย `decision_function` และความน่าจะเป็นด้วย `predict_proba` ซึ่งคือซอฟต์แมกซ์ของลอจิต ตรวจกับลอจิต $(2.0, 1.0, -1.0)$ ของตัวอย่างที่ 6.11 และคำนวณตัววัดรายคลาสจากเมทริกซ์ความสับสนของตัวอย่างที่ 6.13

In [ ]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())                    # ลบค่าสูงสุดก่อนเพื่อไม่ให้ exp ล้น
    return e / e.sum()


p = softmax([2.0, 1.0, -1.0])
print("ตัวอย่างที่ 6.11 p =", p.round(3), "| L ถ้ารอยร้าวจริง =", round(-np.log(p[1]), 2), "| p_รอยร้าว + p_ดับ =", round(p[1] + p[2], 3))
compare("p_ปกติ", p[0], 0.705, 5e-4)
compare("ค่าสูญเสีย -ln p_รอยร้าว", -np.log(p[1]), 1.35, 5e-3)

head = models[best_name]
one_crack = test_idx[patch_labels[test_idx] == 2][0]
logits = head.decision_function(backbone_feats[[one_crack]])[0]
print("\nภาพแตกหนึ่งภาพในชุดทดสอบ: ลอจิต", logits.round(2), "→ softmax", softmax(logits).round(3),
      "| predict_proba", head.predict_proba(backbone_feats[[one_crack]])[0].round(3))


def per_class_metrics(cm, class_names):
    """ความแม่นยำ ความระลึก และ F1 รายคลาส จากเมทริกซ์ความสับสน (แถว = จริง, คอลัมน์ = ทำนาย)"""
    cm = np.asarray(cm)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    return pd.DataFrame({"precision": tp / (tp + fp), "recall": tp / (tp + fn),
                         "F1": 2 * tp / (2 * tp + fp + fn), "support": cm.sum(axis=1)}, index=class_names)


EX613_CM = np.array([[18, 2, 0], [3, 11, 1], [0, 2, 8]])
ex613 = per_class_metrics(EX613_CM, CLASSES)
display(ex613.round(3))
compare("ความถูกต้อง", np.trace(EX613_CM) / EX613_CM.sum(), 0.822, 5e-4)
compare("Macro-F1", ex613["F1"].mean(), 0.818, 5e-4)

### 5.6 การเพิ่มข้อมูลภาพ (Data Augmentation)

การแปลงที่ไม่เปลี่ยนป้ายกำกับของงานหนึ่ง อาจเปลี่ยนป้ายของอีกงานหนึ่ง (ตารางที่ 6.5) ด้านล่างเปรียบเทียบการพลิก/หมุนภาพรอยร้าว (ป้ายยังเป็น "แตก") กับการพลิกภาพตัวเลขบนหน้าจอมิเตอร์ (เลข 2 ที่พลิกซ้ายขวาไม่ใช่เลข 2 อีกต่อไป)

In [ ]:
crack_example = patches[np.flatnonzero(patch_labels == 2)[3]]
digit_two = meter[8:48, 8:32]
views = [("cracked patch", crack_example), ("flip left-right", cv2.flip(crack_example, 1)),
         ("rotate 90", cv2.rotate(crack_example, cv2.ROTATE_90_CLOCKWISE)),
         ("darker (x0.6)", np.clip(crack_example * 0.6, 0, 255).astype(np.uint8)),
         ("digit '2'", digit_two), ("digit flipped", cv2.flip(digit_two, 1))]
fig, axes = plt.subplots(1, len(views), figsize=(13, 2.4))
for ax, (title, img) in zip(axes, views):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

# ฝึกส่วนหัวใหม่ด้วยภาพฝึกที่เพิ่มการพลิกและหมุน 90° (×4) แล้ววัดผลบนชุดทดสอบเดิม
augmented_images = np.concatenate([patches[train_idx], patches[train_idx][:, :, ::-1],
                                   patches[train_idx][:, ::-1, :], np.rot90(patches[train_idx], axes=(1, 2))])
augmented_labels = np.tile(patch_labels[train_idx], 4)
augmented_feats = np.array([frozen_backbone(np.ascontiguousarray(img)) for img in augmented_images])
aug_head = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(augmented_feats, augmented_labels)
aug_pred = aug_head.predict(backbone_feats[test_idx])
print(f"ชุดฝึก {len(train_idx)} → {len(augmented_labels)} ภาพหลังเพิ่มข้อมูล")
print(f"macro-F1 บนชุดทดสอบ: ไม่เพิ่มข้อมูล {results[-1]['macro-F1']:.3f} | เพิ่มข้อมูล {f1_score(patch_labels[test_idx], aug_pred, average='macro'):.3f}")

บนภาพจำลองชุดนี้ซึ่งรอยร้าวมีทุกทิศอยู่แล้ว การเพิ่มข้อมูลเปลี่ยนผลเพียงเล็กน้อย ประโยชน์ของการเพิ่มข้อมูลจะชัดเจนเมื่อชุดฝึกเล็กมาก หรือเมื่อภาพใช้งานจริงมีความหลากหลาย (มุมกล้อง แสง) มากกว่าภาพฝึก

### 5.7 (ทางเลือก) Transfer Learning จริงด้วย ResNet-18 ที่ฝึกกับ ImageNet

เซลล์ถัดไปทำตามขั้นตอน TRANSFER-LEARNING ในหัวข้อ 6.3.2 กับภาพจำลองชุดเดียวกัน: โหลด ResNet-18 ที่ฝึกกับ ImageNet (`torchvision`) แช่แข็งส่วนหลัก ใส่ส่วนหัวใหม่ 3 คลาส ตรวจว่าพารามิเตอร์ที่ฝึกได้เท่ากับ 1,539 ตัว (ตัวอย่างที่ 6.12) ทำซ้ำช่องเดียวเป็นสามช่องและปรับค่าด้วยค่าเฉลี่ย/ส่วนเบี่ยงเบนมาตรฐานของ ImageNet แล้วเปรียบเทียบ (ก) การสกัดคุณลักษณะ (ข) การสกัดคุณลักษณะ + การเพิ่มข้อมูลภาพ และ (ค) การปรับละเอียดชั้นสุดท้าย (`layer4`) ด้วยอัตราการเรียนรู้ที่เล็กกว่า 10 เท่า

- ต้องมี PyTorch + torchvision (Colab มีให้) และอินเทอร์เน็ตเพื่อดาวน์โหลดน้ำหนักราว 45 MB
- ใช้เวลาราว 1 นาทีบน GPU หรือหลายนาทีบน CPU **ตั้ง `RUN_PRETRAINED = True` เพื่อรัน**
- ถ้าต้องการลองกับภาพจริง ให้ดาวน์โหลดชุดข้อมูล ELPV (Deitsch et al., 2019; https://github.com/zae-bayern/elpv-dataset) แล้วแทน `patches` และ `patch_labels` ด้วยภาพและป้ายของชุดนั้น (ระวังเรื่องการแบ่งข้อมูลตามแผง)

In [ ]:
RUN_PRETRAINED = False   # เปลี่ยนเป็น True บน Colab (แนะนำให้เปิด GPU) เพื่อรันส่วนทางเลือกทั้งหมด


def pytorch_available():
    try:
        import torch          # noqa: F401
        import torchvision    # noqa: F401
        return True
    except ImportError:
        return False


if not RUN_PRETRAINED:
    print("ข้ามส่วนทางเลือก 5.7 (RUN_PRETRAINED = False)")
elif not pytorch_available():
    print("ไม่พบ PyTorch/torchvision: ติดตั้งด้วย pip install torch torchvision หรือรันบน Google Colab")
else:
    import copy
    import torch
    from torch import nn
    from torchvision.models import ResNet18_Weights, resnet18

    torch.manual_seed(RANDOM_SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def to_input(images, size=96):
        """uint8 (N, H, W) → เทนเซอร์ (N, 3, size, size) ที่ปรับค่าแบบ ImageNet"""
        x = torch.tensor(images, dtype=torch.float32).unsqueeze(1) / 255.0
        x = x.repeat(1, 3, 1, 1)                                      # ทำซ้ำช่องเดียวเป็นสามช่อง
        x = nn.functional.interpolate(x, size=size, mode="bilinear", align_corners=False)
        return (x - IMAGENET_MEAN) / IMAGENET_STD

    def random_flip_rotate(x):
        """การเพิ่มข้อมูลภาพแบบสุ่มทีละภาพในชุดย่อย: พลิกซ้ายขวา พลิกบนล่าง และหมุน 90° (สุ่มแยกกันทุกภาพ)"""
        out = []
        for img in x:                                              # img มีขนาด (C, H, W)
            if torch.rand(1).item() < 0.5:
                img = torch.flip(img, dims=[2])
            if torch.rand(1).item() < 0.5:
                img = torch.flip(img, dims=[1])
            img = torch.rot90(img, k=int(torch.randint(0, 4, (1,)).item()), dims=[1, 2])
            out.append(img)
        return torch.stack(out)

    base_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)     # ดาวน์โหลดน้ำหนักครั้งแรก

    def make_feature_extractor():
        model = copy.deepcopy(base_model)
        for param in model.parameters():
            param.requires_grad = False                               # แช่แข็งส่วนหลัก
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))      # ส่วนหัวใหม่ (ฝึกได้)
        return model.to(device)

    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total, trainable

    def train(model, learning_rate, epochs, augment):
        x_train, y_train = to_input(patches[train_idx]), torch.tensor(patch_labels[train_idx])
        optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
        loss_fn = nn.CrossEntropyLoss()
        model.eval()   # ใช้ค่าสถิติ BatchNorm ของ ImageNet ตลอด (เหมาะกับข้อมูลน้อย) เกรเดียนต์ยังไหลตามปกติ
        for _ in range(epochs):
            order = torch.randperm(len(x_train))
            for start in range(0, len(order), 32):
                batch = order[start:start + 32]
                xb = x_train[batch]
                if augment:
                    xb = random_flip_rotate(xb)
                optimizer.zero_grad()
                loss = loss_fn(model(xb.to(device)), y_train[batch].to(device))
                loss.backward()
                optimizer.step()
        return model

    def evaluate(model):
        model.eval()
        with torch.no_grad():
            pred = model(to_input(patches[test_idx]).to(device)).argmax(dim=1).cpu().numpy()
        truth = patch_labels[test_idx]
        return {"accuracy": np.mean(pred == truth), "macro-F1": f1_score(truth, pred, average="macro"),
                "recall (cracked)": np.mean(pred[truth == 2] == 2)}

    tl_rows = []
    model_a = make_feature_extractor()
    total, trainable = count_parameters(model_a)
    print(f"พารามิเตอร์ทั้งหมด {total:,} | ที่ฝึกได้ {trainable:,} (ตัวอย่างที่ 6.12: 1,539)")
    train(model_a, 1e-3, epochs=8, augment=False)
    tl_rows.append({"strategy": "(a) feature extraction", "trainable params": trainable, **evaluate(model_a)})

    model_b = train(make_feature_extractor(), 1e-3, epochs=8, augment=True)
    tl_rows.append({"strategy": "(b) feature extraction + augmentation", "trainable params": trainable, **evaluate(model_b)})

    for param in model_a.layer4.parameters():                         # ปลดการแช่แข็งชั้นท้าย
        param.requires_grad = True
    train(model_a, 1e-4, epochs=4, augment=True)                      # อัตราการเรียนรู้เล็กกว่า 10 เท่า
    tl_rows.append({"strategy": "(c) + fine-tune layer4", "trainable params": count_parameters(model_a)[1], **evaluate(model_a)})
    display(pd.DataFrame(tl_rows).set_index("strategy").round(3))

## ส่วนที่ 6 การตรวจจับวัตถุและการแบ่งส่วนภาพ

(หนังสือหัวข้อ 6.4)

### 6.1 รูปแบบกรอบและ IoU (ตัวอย่างที่ 6.14)

กรอบแบบมุม $(x_{\min}, y_{\min}, x_{\max}, y_{\max})$ ใช้ $x$ เป็นแนวนอน (คอลัมน์) และ $y$ เป็นแนวตั้ง (แถว) ส่วนป้ายกำกับแบบ YOLO เก็บจุดศูนย์กลาง ความกว้าง และความสูงที่หารด้วยขนาดภาพแล้ว

In [ ]:
def corners_to_yolo(box, image_width, image_height):
    x_min, y_min, x_max, y_max = box
    return ((x_min + x_max) / 2 / image_width, (y_min + y_max) / 2 / image_height,
            (x_max - x_min) / image_width, (y_max - y_min) / image_height)


def yolo_to_corners(label, image_width, image_height):
    xc, yc, w, h = label
    return ((xc - w / 2) * image_width, (yc - h / 2) * image_height,
            (xc + w / 2) * image_width, (yc + h / 2) * image_height)


def box_iou(a, b):
    """Intersection over Union ของกรอบแบบมุมสองกรอบ"""
    overlap_x = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    overlap_y = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = overlap_x * overlap_y
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return intersection / (area_a + area_b - intersection)


label = corners_to_yolo((200, 120, 320, 360), 640, 480)
print("ป้าย YOLO ของกรอบ (200, 120, 320, 360) ในภาพ 640×480:", label, "→ แปลงกลับ", yolo_to_corners(label, 640, 480))
compare("IoU ลูกถ้วย (กรณี ก)", box_iou((100, 80, 200, 240), (120, 100, 220, 250)), 0.566, 5e-4)
compare("IoU สลักล็อก 20×20 เลื่อน 5 px", box_iou((0, 0, 20, 20), (5, 5, 25, 25)), 0.391, 5e-4)
compare("IoU คอนเสา 200×200 เลื่อน 5 px", box_iou((0, 0, 200, 200), (5, 5, 205, 205)), 0.906, 5e-4)

shifts = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for size, color in [(20, COLORS["orange"]), (50, COLORS["green"]), (200, COLORS["blue"])]:
    ax.plot(shifts, [box_iou((0, 0, size, size), (s, s, size + s, size + s)) for s in shifts],
            color=color, marker="o", ms=3, label=f"{size} x {size} px object")
ax.axhline(0.5, color=COLORS["gray"], ls="--", label="IoU = 0.5")
ax.set_xlabel("prediction shifted right and down by (px)")
ax.set_ylabel("IoU")
ax.legend()
ax.set_title("Example 6.14: small objects are sensitive to localization error")
plt.show()

### 6.2 ถอดรหัสผลลัพธ์ของ YOLO หนึ่งช่อง (ตัวอย่างที่ 6.15)

In [ ]:
def decode_yolo_cell(row, col, x, y, w, h, confidence, class_probs, image_size=448, grid=7):
    """แปลงผลลัพธ์ของกรอบหนึ่งกรอบจากช่อง (row, col) เป็นกรอบแบบมุม (พิกเซล) และคะแนนเฉพาะคลาส"""
    cell = image_size / grid
    xc, yc = (col + x) * cell, (row + y) * cell          # x, y เทียบกับมุมของช่อง
    bw, bh = w * image_size, h * image_size              # w, h เทียบกับขนาดภาพ
    box = (xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2)
    return box, confidence * np.asarray(class_probs)


YOLO_CLASSES = ["insulator OK", "insulator broken", "bird nest"]
box, class_scores = decode_yolo_cell(3, 2, 0.5, 0.25, 0.2, 0.4, 0.8, [0.10, 0.85, 0.05])
print("กรอบ (x_min, y_min, x_max, y_max):", tuple(round(v, 1) for v in box), "(หนังสือ 115.2, 118.4, 204.8, 297.6)")
print("คะแนนเฉพาะคลาส:", dict(zip(YOLO_CLASSES, class_scores.round(2))))
print("ผ่านเกณฑ์คะแนน 0.25:", [c for c, s in zip(YOLO_CLASSES, class_scores) if s >= 0.25])
B, C, S = 2, len(YOLO_CLASSES), 7
print(f"ขนาดเทนเซอร์ผลลัพธ์ S × S × (5B + C) = {S} × {S} × {5 * B + C} = {S * S * (5 * B + C)} ค่า")

### 6.3 ตัวตรวจจับอย่างง่ายบนภาพจากโดรนจำลอง

สร้างภาพจำลอง 12 ภาพ ภาพละ 6 วัตถุสว่าง (จานลูกถ้วยรูปวงรี) บนฟ้าที่ไล่ระดับความสว่าง มีจุดสว่างเล็ก ๆ (แสงสะท้อน นก) เป็นตัวหลอก และบางวัตถุจางจนกลืนกับฟ้า แล้วใช้ตัวตรวจจับแบบดั้งเดิมจากส่วนที่ 4: ลบพื้นหลัง (ประมาณด้วยการเบลอแรง ๆ เพราะฟ้าสว่างไม่เท่ากัน) → กรองเกาส์เซียน → Otsu → องค์ประกอบเชื่อมต่อ → กรอบ โดยคะแนนความมั่นใจคำนวณจากความเปรียบต่างและความทึบของวัตถุ

เพื่อให้เห็นปัญหา "กรอบซ้ำ" แบบที่ตัวตรวจจับจริงมี ตัวตรวจจับนี้รันที่ความเบลอ 3 ระดับ ($\sigma$ = 1, 2, 3) แล้วรวมกรอบทั้งหมด วัตถุหนึ่งชิ้นจึงได้กรอบหลายกรอบที่ซ้อนกัน ซึ่งต้องใช้ NMS (แบบฝึกปฏิบัติข้อ 3) ก่อนวัดผลด้วย AP (แบบฝึกปฏิบัติข้อ 4)

In [ ]:
def make_drone_scene(seed, height=240, width=320, n_objects=6):
    """ภาพจากโดรนจำลอง คืน (ภาพ uint8, รายการกรอบจริงแบบมุม)"""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:height, 0:width]
    image = (70 + 50 * yy / height + 10 * np.sin(xx / 40) + rng.normal(0, 6, (height, width))).astype(np.float32)
    boxes = []
    for _ in range(500):
        if len(boxes) == n_objects:
            break
        bw, bh = int(rng.integers(14, 44)), int(rng.integers(14, 44))
        x0, y0 = int(rng.integers(2, width - bw - 2)), int(rng.integers(2, height - bh - 2))
        box = (x0, y0, x0 + bw, y0 + bh)
        too_close = any(not (box[2] + 6 < b[0] or b[2] + 6 < box[0] or box[3] + 6 < b[1] or b[3] + 6 < box[1]) for b in boxes)
        if too_close:
            continue
        boxes.append(box)
        cv2.ellipse(image, (x0 + bw // 2, y0 + bh // 2), (bw // 2, bh // 2), 0, 0, 360, float(rng.uniform(120, 215)), -1)
    for _ in range(25):                                   # จุดสว่างเล็ก ๆ ที่เป็นตัวหลอก
        cv2.circle(image, (int(rng.integers(0, width)), int(rng.integers(0, height))),
                   int(rng.integers(1, 3)), float(rng.uniform(140, 200)), -1)
    return np.clip(image, 0, 255).astype(np.uint8), boxes


def detect_bright_objects(image, sigma, min_area=15):
    """ตัวตรวจจับแบบดั้งเดิม: ลบพื้นหลัง → เบลอ → Otsu → องค์ประกอบเชื่อมต่อ คืนรายการ (กรอบ, คะแนน)"""
    # ฟ้าสว่างไม่เท่ากันทั้งภาพ จึงประมาณพื้นหลังด้วยการเบลอแรง ๆ แล้วลบออกก่อนหาค่าเกณฑ์
    background = cv2.GaussianBlur(image, (0, 0), 25).astype(np.float32)
    flat = np.clip(image.astype(np.float32) - background + 60, 0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(flat, (0, 0), sigma)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    detections = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < min_area:
            continue
        contrast = flat[labels == i].mean() - 60          # ความสว่างเหนือพื้นหลัง
        fill = area / (w * h)                             # วงรีเต็มกรอบ ≈ π/4 = 0.785
        score = float(np.clip(contrast / 100, 0, 1) * min(1.0, fill / 0.785))
        detections.append(((int(x), int(y), int(x + w), int(y + h)), round(score, 4)))
    return detections


N_SCENES = 12
scenes, ground_truths, candidates = {}, {}, []
for scene_id in range(N_SCENES):
    image, gt_boxes = make_drone_scene(seed=RANDOM_SEED + scene_id)
    scenes[scene_id], ground_truths[scene_id] = image, gt_boxes
    for sigma in (1, 2, 3):
        for box, score in detect_bright_objects(image, sigma):
            candidates.append({"image_id": scene_id, "box": box, "score": score})
n_gt = sum(len(b) for b in ground_truths.values())
print(f"ภาพ {N_SCENES} ภาพ | วัตถุจริง {n_gt} ชิ้น | กรอบผู้สมัครจากตัวตรวจจับ {len(candidates)} กรอบ (มีกรอบซ้ำ)")

scene0 = [c for c in candidates if c["image_id"] == 0]
iou_table = pd.DataFrame([[box_iou(c["box"], g) for g in ground_truths[0]] for c in scene0],
                         index=[f"det {i} (s={c['score']:.2f})" for i, c in enumerate(scene0)],
                         columns=[f"GT {j}" for j in range(len(ground_truths[0]))])
display(iou_table.round(2))

fig, ax = plt.subplots(figsize=(7, 5.2))
ax.imshow(scenes[0], cmap="gray")
for g in ground_truths[0]:
    ax.add_patch(plt.Rectangle((g[0], g[1]), g[2] - g[0], g[3] - g[1], fill=False, ec=COLORS["green"], lw=2))
for c in scene0:
    b = c["box"]
    ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=COLORS["orange"], lw=1, ls="--"))
ax.set_title("scene 0: ground truth (green) and candidate boxes (orange, 3 blur levels)")
ax.axis("off")
plt.show()

ตาราง IoU แสดงว่าวัตถุจริงหนึ่งชิ้นมีกรอบผู้สมัครหลายกรอบที่ IoU สูงมาก (กรอบซ้ำ) ถ้านำไปนับตรง ๆ จะได้ FP จำนวนมาก บางแถวมี IoU เป็นศูนย์ทุกคอลัมน์ ซึ่งคือการเตือนผิดจากจุดสว่างที่เป็นตัวหลอก

**คำถามชวนคิด** ถ้าจานลูกถ้วยสองจานอยู่ชิดกันจนกรอบซ้อนกัน ตัวตรวจจับแบบองค์ประกอบเชื่อมต่อนี้จะให้ผลอย่างไร และ YOLO รุ่นแรกมีข้อจำกัดแบบเดียวกันหรือไม่ (ดูตีความของตัวอย่างที่ 6.15)

### 6.4 วัดผลการแบ่งส่วนภาพด้วย IoU และ Dice (ตัวอย่างที่ 6.18)

In [ ]:
def segmentation_metrics(truth, pred):
    """ความถูกต้องรายพิกเซล IoU และ Dice ของคลาสที่สนใจ"""
    truth, pred = np.asarray(truth, bool), np.asarray(pred, bool)
    tp = np.sum(truth & pred)
    fp = np.sum(~truth & pred)
    fn = np.sum(truth & ~pred)
    tn = np.sum(~truth & ~pred)
    iou = tp / (tp + fp + fn) if tp + fp + fn else 1.0
    dice = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn, "pixel accuracy": (tp + tn) / truth.size, "IoU": iou, "Dice": dice}


EX618_TRUTH = np.zeros((6, 6), bool)
for r, c in [(0, 1), (1, 1), (1, 2), (2, 2), (3, 3), (4, 3), (4, 4), (5, 4)]:
    EX618_TRUTH[r, c] = True
EX618_PRED = np.zeros((6, 6), bool)
for r, c in [(0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (3, 2), (3, 3), (3, 4), (4, 3), (5, 5)]:
    EX618_PRED[r, c] = True

seg_rows = {"model (Example 6.18)": segmentation_metrics(EX618_TRUTH, EX618_PRED),
            "lazy model (all background)": segmentation_metrics(EX618_TRUTH, np.zeros((6, 6), bool))}

# ขยายไปภาพผิวลูกถ้วยในส่วนที่ 2: แบ่งส่วนรอยร้าวด้วยค่าเกณฑ์อย่างง่ายบนภาพที่กรองมัธยฐานแล้ว
denoised = cv2.medianBlur(noisy_surface, 3)
seg_rows["threshold < 120 on median-filtered surface"] = segmentation_metrics(surface_crack, denoised < 120)
seg_rows["lazy model on surface"] = segmentation_metrics(surface_crack, np.zeros_like(surface_crack))
display(pd.DataFrame(seg_rows).T.round(3))
compare("IoU ตัวอย่างที่ 6.18", seg_rows["model (Example 6.18)"]["IoU"], 0.5, 1e-9)
compare("Dice ตัวอย่างที่ 6.18", seg_rows["model (Example 6.18)"]["Dice"], 0.667, 5e-4)
print(f"รอยร้าวกินพื้นที่ {surface_crack.mean() * 100:.1f}% ของภาพผิวลูกถ้วย")

โมเดลขี้เกียจบนภาพผิวลูกถ้วยได้ความถูกต้องรายพิกเซลเกือบ 0.99 ทั้งที่ไม่พบรอยร้าวเลย เพราะรอยร้าวกินพื้นที่เพียงราวร้อยละ 1–2 ของภาพ งานแบ่งส่วนวัตถุขนาดเล็กจึงต้องรายงาน IoU หรือ Dice ของคลาสที่สนใจ

**การนำไปใช้ (ตัวอย่างที่ 6.19)** เมื่อได้หน้ากากแล้ว ร้อยละของพื้นที่คำนวณได้ทันที เช่น `mask.sum() / panel_mask.sum() * 100` แต่ต้องจำไว้ว่าร้อยละของพื้นที่คราบสกปรกบนแผงโซลาร์เซลล์ไม่เท่ากับร้อยละของกำลังไฟฟ้าที่ลดลง

### 6.5 (ทางเลือก) ตัวตรวจจับวัตถุที่ฝึกไว้แล้ว

เซลล์ถัดไปรันตัวตรวจจับ Faster R-CNN (ส่วนหลัก MobileNetV3) ที่ฝึกกับชุดข้อมูล COCO จาก `torchvision` บนภาพตัวอย่างของ scikit-image (`skimage.data.astronaut()` ภาพสาธารณสมบัติจาก NASA) โดยปิด NMS ภายในโมเดล แล้วลองเกณฑ์คะแนนและเกณฑ์ IoU ของ NMS หลายค่าเพื่อดูว่าจำนวนกรอบเปลี่ยนอย่างไร ใช้ตัวแปร `RUN_PRETRAINED` เดียวกับส่วนที่ 5.7 ถ้าต้องการลองกับภาพของตนเอง ให้อัปโหลดภาพเข้า Colab แล้วกำหนด `IMAGE_PATH`

In [ ]:
IMAGE_PATH = None   # เช่น "my_insulator.jpg" (ไฟล์ที่อัปโหลดเข้า Colab) ถ้าเป็น None จะใช้ภาพตัวอย่าง

if not RUN_PRETRAINED:
    print("ข้ามส่วนทางเลือก 6.5 (RUN_PRETRAINED = False)")
elif not pytorch_available():
    print("ไม่พบ PyTorch/torchvision: ติดตั้งด้วย pip install torch torchvision หรือรันบน Google Colab")
else:
    import torch
    from torchvision.models.detection import (FasterRCNN_MobileNet_V3_Large_FPN_Weights,
                                              fasterrcnn_mobilenet_v3_large_fpn)
    from torchvision.ops import nms as torchvision_nms

    if IMAGE_PATH is None:
        from skimage import data
        photo = data.astronaut()                                          # RGB uint8
    else:
        photo = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)  # imread ให้ BGR
    weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
    detector = fasterrcnn_mobilenet_v3_large_fpn(weights=weights, box_score_thresh=0.05, box_nms_thresh=1.0).eval()
    with torch.no_grad():
        output = detector([weights.transforms()(torch.from_numpy(photo).permute(2, 0, 1))])[0]
    categories = weights.meta["categories"]

    count_rows = []
    for score_threshold in (0.05, 0.3, 0.7):
        for tau in (0.3, 0.5, 0.8):
            keep = output["scores"] >= score_threshold
            boxes, scores, labels = output["boxes"][keep], output["scores"][keep], output["labels"][keep]
            kept = torchvision_nms(boxes, scores, tau)                    # NMS ไม่แยกคลาส เพื่อความเรียบง่าย
            count_rows.append({"score threshold": score_threshold, "NMS tau": tau, "boxes before NMS": len(boxes),
                               "boxes after NMS": len(kept)})
    display(pd.DataFrame(count_rows))

    keep = output["scores"] >= 0.5
    kept = torchvision_nms(output["boxes"][keep], output["scores"][keep], 0.5)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(photo)
    for i in kept:
        x0, y0, x1, y1 = output["boxes"][keep][i].tolist()
        name = categories[output["labels"][keep][i]]
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=COLORS["orange"], lw=2))
        ax.text(x0, y0 - 3, f"{name} {output['scores'][keep][i]:.2f}", color="white", fontsize=8,
                bbox=dict(facecolor=COLORS["orange"], alpha=0.8, pad=1))
    ax.set_title("pretrained detector: score >= 0.5, NMS tau = 0.5")
    ax.axis("off")
    plt.show()

## ส่วนที่ 7 การประยุกต์ในอุตสาหกรรม

(หนังสือหัวข้อ 6.5)

### 7.1 เกณฑ์ของระบบตรวจหาค่าผิดปกติและผลของอัตราฐาน (ตัวอย่างที่ 6.20–6.21)

In [ ]:
good_scores = np.array([0.20, 0.22, 0.18, 0.25, 0.21, 0.19, 0.23, 0.20, 0.17, 0.25])
defect_scores = np.array([0.45, 0.62, 0.30, 0.27, 0.95])
mu, sigma = good_scores.mean(), good_scores.std()          # np.std หารด้วย n ตามข้อตกลงของหนังสือ
anomaly_threshold = mu + 3 * sigma
compare("เกณฑ์ μ + 3σ", anomaly_threshold, 0.288, 5e-4)
print("ชิ้นงานมีตำหนิที่ถูกคัดออก:", defect_scores[defect_scores > anomaly_threshold],
      "| ความระลึก =", np.mean(defect_scores > anomaly_threshold))


def inspection_load(parts_per_hour, defect_rate, recall, false_positive_rate, hours=8):
    """ภาระของสถานีตรวจซ้ำต่อกะ (ตัวอย่างที่ 6.21)"""
    bad = parts_per_hour * defect_rate
    good = parts_per_hour - bad
    tp, fp = recall * bad, false_positive_rate * good
    return {"rejected per shift": (tp + fp) * hours, "good parts rejected": fp * hours,
            "defects escaped": (bad - tp) * hours, "precision": tp / (tp + fp)}


load_table = pd.DataFrame({f"FPR = {fpr:.1%}": inspection_load(3600, 0.005, 0.95, fpr) for fpr in (0.005, 0.01, 0.02)}).T
display(load_table.round(3))
compare("ความแม่นยำที่ FPR 1%", load_table.loc["FPR = 1.0%", "precision"], 0.323, 5e-4)

### 7.2 ภาพถ่ายความร้อนของขั้วเบรกเกอร์สามเฟส (ตัวอย่างที่ 6.22)

สร้างภาพถ่ายความร้อนแบบเรดิโอเมตริกจำลอง (แต่ละพิกเซลเก็บอุณหภูมิเป็น °C โดยตรง ไม่ใช่สี) ของขั้วเบรกเกอร์สามเฟส อุณหภูมิอากาศในตู้ 30 °C แล้วทำตามขั้นตอน THERMAL-INSPECTION อย่างย่อ: หาอุณหภูมิสูงสุดภายในหน้ากากของแต่ละขั้ว → คำนวณ ΔT → จัดระดับตามตารางที่ 6.8 → ปรับตามกระแสด้วยการประมาณ $I^2$

> **ความปลอดภัย** การถ่ายภาพความร้อนในตู้ไฟฟ้าที่มีไฟต้องทำโดยผู้ได้รับอนุญาตและสวมอุปกรณ์ป้องกันตามข้อกำหนดของหน่วยงาน ตัวเลขในส่วนนี้เป็นค่าสมมติตามหนังสือ

In [ ]:
def make_thermal_image(peak_temps, ambient=30.0, seed=RANDOM_SEED, height=90, width=150):
    """ภาพอุณหภูมิจำลอง (°C) ของขั้วสามขั้ว คืน (ภาพ float, หน้ากากของแต่ละขั้ว)"""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:height, 0:width]
    image = ambient + rng.normal(0, 0.3, (height, width))
    masks = {}
    for k, (phase, peak) in enumerate(peak_temps.items()):
        cx, cy = 25 + 50 * k, 45
        mask = (np.abs(xx - cx) <= 12) & (np.abs(yy - cy) <= 25)
        hot = np.exp(-((xx - cx) ** 2 + (yy - (cy - 15)) ** 2) / (2 * 6.0 ** 2))  # จุดร้อนที่จุดต่อด้านบน
        terminal = ambient + 3 + (peak - ambient - 3) * hot + rng.normal(0, 0.2, (height, width)) * (1 - hot)
        image = np.where(mask, np.minimum(terminal, peak), image)       # ค่าสูงสุดของขั้วเท่ากับ peak พอดี
        masks[phase] = mask
    return image, masks


def neta_action(delta_similar=None, delta_ambient=None):
    """ระดับการดำเนินการตามตารางที่ 6.8 (ใช้ระดับที่รุนแรงกว่าเมื่อสองเกณฑ์ให้ผลต่างกัน)"""
    names = ["อาจมีความบกพร่อง ควรตรวจสอบเพิ่มเติม", "น่าจะมีความบกพร่อง ซ่อมเมื่อมีโอกาส",
             "เฝ้าติดตามจนกว่าจะแก้ไขได้", "ความบกพร่องรุนแรง ซ่อมทันที"]   # ข้อความตามตารางที่ 6.8
    levels = []
    if delta_similar is not None:
        levels.append(3 if delta_similar > 15 else 1 if delta_similar >= 4 else 0)
    if delta_ambient is not None:
        levels.append(3 if delta_ambient > 40 else 2 if delta_ambient >= 21 else 1 if delta_ambient >= 11 else 0)
    return names[max(levels)]


AMBIENT = 30.0
thermal, terminal_masks = make_thermal_image({"A": 41.0, "B": 43.0, "C": 61.0}, ambient=AMBIENT)
currents = {"A": 100, "B": 102, "C": 130}
t_max = {phase: thermal[mask].max() for phase, mask in terminal_masks.items()}

hottest = max(t_max, key=t_max.get)
others = {p: t for p, t in t_max.items() if p != hottest}
delta_similar = t_max[hottest] - max(others.values())
delta_ambient = t_max[hottest] - AMBIENT
print("T_max ของแต่ละขั้ว (°C):", {p: round(t, 1) for p, t in t_max.items()})
print(f"เฟส {hottest}: ΔT เทียบเฟสข้างเคียง = {delta_similar:.1f} °C | ΔT เทียบอากาศ = {delta_ambient:.1f} °C → {neta_action(delta_similar, delta_ambient)}")

# ขั้นที่ 3–4 ของตัวอย่างที่ 6.22: อุณหภูมิที่ควรเป็นถ้าร้อนเพราะโหลดอย่างเดียว (แปรตาม I²)
rise_others = np.mean([t_max[p] - AMBIENT for p in others])
current_others = np.mean([currents[p] for p in others])
expected = AMBIENT + rise_others * (currents[hottest] / current_others) ** 2
print(f"อุณหภูมิที่คาดจากโหลด {expected:.1f} °C | ส่วนเกินที่อธิบายไม่ได้ {t_max[hottest] - expected:.1f} °C (หนังสือ 49.9 และ 11.1)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, cmap in zip(axes, ["inferno", "gray"]):
    im = ax.imshow(thermal, cmap=cmap)
    plt.colorbar(im, ax=ax, label="temperature (C)")
    ax.axis("off")
axes[0].set_title("pseudo-color palette (for humans)")
axes[1].set_title("same radiometric data in gray")
for phase, mask in terminal_masks.items():
    r, c = np.argwhere(mask).mean(axis=0)
    axes[0].text(c, mask.shape[0] - 5, f"{phase}: {t_max[phase]:.0f} C", color="white", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

ภาพสองภาพด้านบนมาจากข้อมูลอุณหภูมิชุดเดียวกัน ต่างกันเพียงจานสี การวิเคราะห์จึงต้องใช้ค่าอุณหภูมิดิบ (`thermal`) ไม่ใช่สี RGB ของภาพสีเทียม AI ช่วย "หาและจัดระเบียบ" ชิ้นส่วน (ในที่นี้คือหน้ากากของขั้ว) ส่วนการตัดสินความรุนแรงยังใช้เกณฑ์ทางวิศวกรรมที่อธิบายได้

### ใช้โค้ดใน notebook นี้ตรวจคำตอบแบบฝึกหัดท้ายบท

ฟังก์ชันที่สร้างไว้ใช้ตรวจคำตอบเชิงตัวเลขของแบบฝึกหัดบทที่ 6 ได้ เช่น `image_bytes` และ `gsd_mm` (ข้อ 1–2) `rgb_to_gray` และ `rgb_to_hsv_deg` (ข้อ 3) `filter2d` และ `median_filter` (ข้อ 5–6 ใช้ `padding` ใดก็ได้ แล้วอ่านเฉพาะตำแหน่งที่เคอร์เนลวางได้ครบ) `otsu_table` (ข้อ 7) `cv2.connectedComponents` และฟังก์ชันในแบบฝึกปฏิบัติข้อ 2 (ข้อ 8) `conv_parameters` และ `softmax` (ข้อ 9–11) `box_iou` `corners_to_yolo` `decode_yolo_cell` (ข้อ 13–14) ฟังก์ชันในแบบฝึกปฏิบัติข้อ 3–4 (ข้อ 14–15) `segmentation_metrics` (ข้อ 16) และ `neta_action` (ข้อ 18) ควรคำนวณด้วยมือก่อน แล้วจึงใช้โค้ดตรวจ

## ส่วนที่ 8 แบบฝึกปฏิบัติ

ทำข้อ 1–4 โดยเติมโค้ดในเซลล์ที่มี `# TODO` แล้วรันเซลล์ตรวจคำตอบที่อยู่ถัดไป (ไม่ต้องแก้เซลล์ตรวจ) และตอบคำถามวิเคราะห์ในข้อ 5

### ข้อ 1 วิธีของ Otsu สำหรับภาพ 8 บิต

เขียน `otsu_threshold(image)` ที่รับภาพระดับเทา `uint8` แล้วคืนค่าเกณฑ์ $t^*$ (จำนวนเต็ม 0–254) ที่ทำให้ $\sigma_B^2 = \omega_0\omega_1(\mu_0 - \mu_1)^2$ สูงสุด โดยกลุ่มมืดคือค่า $\le t$ ถ้าเสมอให้เลือก $t$ ที่น้อยที่สุด **ห้ามใช้** `cv2.threshold`

คำแนะนำ: หาฮิสโทแกรม 256 ระดับด้วย `np.bincount(image.ravel(), minlength=256)` แล้วใช้ `np.cumsum` คำนวณ $N_0$ และผลรวม $\sum k \cdot n_k$ ของกลุ่มมืดสำหรับทุก $t$ พร้อมกัน (ข้าม $t$ ที่กลุ่มใดกลุ่มหนึ่งว่าง)

In [ ]:
def otsu_threshold(image):
    hist = np.bincount(np.asarray(image, dtype=np.uint8).ravel(), minlength=256).astype(float)
    # TODO: คำนวณ N0, N1, mu0, mu1 ของทุก t ด้วย np.cumsum → sigma_B^2 → คืน t ที่ดีที่สุด (int)
    return None


meter_t = otsu_threshold(meter_blur)

In [ ]:
# เซลล์ตรวจคำตอบข้อ 1 (ไม่ต้องแก้)
if meter_t is None:
    print("ข้อ 1: ยังไม่ได้ทำ")
else:
    checks = {"ภาพมิเตอร์": meter_blur, "ภาพผิวลูกถ้วย": surface, "ภาพโดรน 0": scenes[0]}
    ok = True
    for name, img in checks.items():
        mine = otsu_threshold(img)
        reference = int(cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0])
        ok &= mine == reference
        print(f"{name:14s}: ของคุณ {mine} | cv2 {reference}")
    ex67 = otsu_threshold(meter_pixels)
    print(f"ตัวอย่างที่ 6.7: {ex67} (หนังสือ 3)")
    print("ถูกต้อง" if ok and ex67 == 3 else "ยังไม่ถูก ลองตรวจอีกครั้ง")

### ข้อ 2 เขียนการกร่อนและการพองตัวเอง

เขียน `erode_binary(binary, size=3)` และ `dilate_binary(binary, size=3)` สำหรับภาพสองค่า (0/1) ด้วยส่วนประกอบโครงสร้างสี่เหลี่ยม `size × size` โดยถือว่าพิกเซลนอกภาพเป็น 0 (ข้อตกลงของแบบฝึกหัดท้ายบท) **ห้ามใช้** `cv2.erode` `cv2.dilate` หรือ `scipy.ndimage` จากนั้น `opening` และ `closing` ที่ให้ไว้จะใช้ฟังก์ชันของคุณ

คำแนะนำ: เติมขอบด้วย `np.pad(binary, size // 2)` แล้ววนทุกพิกเซล ใช้ `.all()` สำหรับการกร่อน และ `.any()` สำหรับการพองตัว

In [ ]:
def erode_binary(binary, size=3):
    binary = np.asarray(binary, dtype=np.uint8)
    # TODO: ผลลัพธ์เป็น 1 เมื่อทุกพิกเซลใต้ส่วนประกอบโครงสร้างเป็น 1
    return None


def dilate_binary(binary, size=3):
    binary = np.asarray(binary, dtype=np.uint8)
    # TODO: ผลลัพธ์เป็น 1 เมื่ออย่างน้อยหนึ่งพิกเซลใต้ส่วนประกอบโครงสร้างเป็น 1
    return None


def opening(binary, size=3):
    return dilate_binary(erode_binary(binary, size), size)


def closing(binary, size=3):
    return erode_binary(dilate_binary(binary, size), size)

In [ ]:
# เซลล์ตรวจคำตอบข้อ 2 (ไม่ต้องแก้)
if erode_binary(case_a) is None or dilate_binary(case_a) is None:
    print("ข้อ 2: ยังไม่ได้ทำ")
else:
    print_binary(opening(case_a), "ตัวอย่างที่ 6.8 กรณี ก หลังเปิด")
    print_binary(closing(case_b), "ตัวอย่างที่ 6.8 กรณี ข หลังปิด")
    rng_check = np.random.default_rng(7)
    random_binary = (rng_check.random((40, 50)) < 0.4).astype(np.uint8)
    agree = True
    for size in (3, 5):
        k = np.ones((size, size), np.uint8)
        ref_erode = cv2.erode(random_binary, k, borderType=cv2.BORDER_CONSTANT, borderValue=0)
        ref_dilate = cv2.dilate(random_binary, k, borderType=cv2.BORDER_CONSTANT, borderValue=0)
        agree &= np.array_equal(erode_binary(random_binary, size), ref_erode)
        agree &= np.array_equal(dilate_binary(random_binary, size), ref_dilate)
    book_ok = opening(case_a).sum() == 9 and closing(case_b).sum() == 10 and closing(case_b)[2:4, 2:7].all()
    print("ตรงกับ cv2 บนภาพสุ่ม (ขนาด 3 และ 5):", agree, "| ตรงกับตัวอย่างที่ 6.8:", bool(book_ok))
    print("ถูกต้อง" if agree and book_ok else "ยังไม่ถูก ลองตรวจอีกครั้ง")

### ข้อ 3 Non-Maximum Suppression

เขียน `nms(boxes, scores, tau)` ตามรหัสเทียม NMS ในหัวข้อ 6.4.1 รับรายการกรอบแบบมุม รายการคะแนน และเกณฑ์ $\tau$ คืน **รายการดัชนี** ของกรอบที่เก็บไว้ เรียงตามคะแนนจากมากไปน้อย โดยลบกรอบเมื่อ IoU กับกรอบที่เก็บแล้ว $\ge \tau$ (ใช้ `box_iou` จากส่วนที่ 6.1) แล้วนำไปใช้กับกรอบผู้สมัคร `candidates` ทีละภาพ เก็บผลไว้ใน `detections_nms` (รายการ dict แบบเดียวกับ `candidates`)

In [ ]:
def nms(boxes, scores, tau=0.5):
    # TODO: เรียงดัชนีตามคะแนนจากมากไปน้อย → เก็บกรอบแรก → ลบกรอบที่ IoU >= tau → ทำซ้ำ
    return None


detections_nms = None   # TODO: ใช้ nms กับ candidates ของแต่ละ image_id ที่ tau = 0.5

In [ ]:
# เซลล์ตรวจคำตอบข้อ 3 (ไม่ต้องแก้)
EX616_BOXES = [(100, 100, 200, 200), (110, 105, 210, 205), (300, 120, 380, 200), (105, 95, 195, 190)]
EX616_SCORES = [0.90, 0.75, 0.80, 0.60]
if nms(EX616_BOXES, EX616_SCORES, 0.5) is None or detections_nms is None:
    print("ข้อ 3: ยังไม่ได้ทำ")
else:
    names = "ABCD"
    kept_05 = [names[i] for i in nms(EX616_BOXES, EX616_SCORES, 0.5)]
    kept_08 = [names[i] for i in nms(EX616_BOXES, EX616_SCORES, 0.8)]
    print("ตัวอย่างที่ 6.16 τ = 0.5:", kept_05, "(หนังสือ A, C) | τ = 0.8:", kept_08)
    agree = True
    for scene_id in range(N_SCENES):                      # เทียบกับ cv2.dnn.NMSBoxes (กรอบแบบ x, y, w, h)
        in_scene = [c for c in candidates if c["image_id"] == scene_id]
        xywh = [[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in (c["box"] for c in in_scene)]
        ref = sorted(np.array(cv2.dnn.NMSBoxes(xywh, [c["score"] for c in in_scene], 0.0, 0.5)).ravel().tolist())
        mine = sorted(nms([c["box"] for c in in_scene], [c["score"] for c in in_scene], 0.5))
        agree &= ref == mine
    print(f"กรอบผู้สมัคร {len(candidates)} → หลัง NMS {len(detections_nms)} กรอบ | วัตถุจริง {n_gt} ชิ้น | ตรงกับ cv2.dnn.NMSBoxes: {agree}")
    print("ถูกต้อง" if kept_05 == ["A", "C"] and "D" in kept_08 and agree else "ยังไม่ถูก ลองตรวจอีกครั้ง")

### ข้อ 4 Average Precision

เขียน `average_precision(detections, ground_truths, tau=0.5)` ตามรหัสเทียม AVERAGE-PRECISION ในหัวข้อ 6.4.1

- `detections` คือรายการ dict ที่มีคีย์ `"image_id"` `"box"` `"score"` และ `ground_truths` คือ dict จาก `image_id` ไปยังรายการกรอบจริง
- เรียงกรอบทำนายตามคะแนนจากมากไปน้อย สำหรับแต่ละกรอบ หากรอบจริงในภาพเดียวกันที่ IoU สูงสุด ถ้า IoU $\ge \tau$ และกรอบจริงนั้นยังไม่ถูกจับคู่ นับเป็น TP มิฉะนั้นเป็น FP
- ใช้ความแม่นยำที่ปรับแล้วแบบทุกจุด $p_{\text{interp}}(r) = \max_{\tilde r \ge r} p(\tilde r)$ แล้ว $\text{AP} = \sum_k (r_k - r_{k-1})\, p_{\text{interp}}(r_k)$
- คืนค่า `(ap, precisions, recalls)` โดย `precisions` และ `recalls` เป็นรายการค่าสะสมหลังแต่ละกรอบ (ตามลำดับคะแนน)

In [ ]:
def average_precision(detections, ground_truths, tau=0.5):
    n_gt = sum(len(boxes) for boxes in ground_truths.values())
    ordered = sorted(detections, key=lambda d: d["score"], reverse=True)
    matched = {image_id: [False] * len(boxes) for image_id, boxes in ground_truths.items()}
    precisions, recalls = [], []
    # TODO: จับคู่ทีละกรอบ → TP/FP สะสม → precisions, recalls → ความแม่นยำที่ปรับแล้ว → AP
    ap = None
    return ap, precisions, recalls


ap_after_nms = None   # TODO: average_precision(detections_nms, ground_truths) แล้วเก็บเฉพาะ AP

In [ ]:
# เซลล์ตรวจคำตอบข้อ 4 (ไม่ต้องแก้)
# สร้างกรณีของตัวอย่างที่ 6.17: ลูกถ้วยแตกจริง 5 ใบ (ภาพละใบ) กรอบทำนาย 6 กรอบ
EX617_GT = {i: [(0, 0, 10, 10)] for i in range(5)}
EX617_DETS = [
    {"image_id": 0, "box": (0, 0, 10, 10), "score": 0.95},   # TP
    {"image_id": 1, "box": (0, 0, 10, 10), "score": 0.90},   # TP
    {"image_id": 2, "box": (50, 50, 60, 60), "score": 0.80},  # FP: ไม่ซ้อนกับกรอบจริง
    {"image_id": 2, "box": (1, 0, 11, 10), "score": 0.70},   # TP
    {"image_id": 0, "box": (0, 1, 10, 11), "score": 0.60},   # FP: กรอบซ้ำของวัตถุที่จับคู่แล้ว
    {"image_id": 3, "box": (1, 1, 11, 11), "score": 0.50},   # TP (IoU = 81/119 ≈ 0.68)
]
result = average_precision(EX617_DETS, EX617_GT)
if result[0] is None or ap_after_nms is None:
    print("ข้อ 4: ยังไม่ได้ทำ")
else:
    ap617, prec617, rec617 = result
    print("ความแม่นยำ:", np.round(prec617, 3), "| ความระลึก:", np.round(rec617, 2))
    print(f"AP ตัวอย่างที่ 6.17 = {ap617:.3f} (หนังสือ 0.683)")
    ap_raw = average_precision(candidates, ground_truths)[0]
    ap_nms, prec, rec = average_precision(detections_nms, ground_truths)
    print(f"\nตัวตรวจจับอย่างง่าย: AP@0.5 ก่อน NMS = {ap_raw:.3f} | หลัง NMS = {ap_nms:.3f}")
    print(f"AP@0.75 หลัง NMS = {average_precision(detections_nms, ground_truths, tau=0.75)[0]:.3f}")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.step(rec, prec, where="post", color=COLORS["blue"])
    ax.set_xlabel("recall")
    ax.set_ylabel("precision")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"simple detector after NMS: AP@0.5 = {ap_nms:.3f}")
    plt.show()
    print("ถูกต้อง" if abs(ap617 - 0.683) < 5e-4 and ap_nms > ap_raw else "ยังไม่ถูก ลองตรวจอีกครั้ง")

### ข้อ 5 วิเคราะห์ผล

1. ในข้อ 1 ถ้าแสงในตู้สว่างขึ้นจนทุกพิกเซลของภาพมิเตอร์เพิ่มขึ้น 20 ระดับ ค่าเกณฑ์ของ Otsu จะเปลี่ยนอย่างไร (ลองทดสอบด้วย `cv2.add(meter_blur, 20)`) และทำไมจึงดีกว่าค่าเกณฑ์ตายตัว แต่ถ้าแสงสว่างไม่เท่ากันทั่วภาพ ควรใช้วิธีใดแทน
2. ในข้อ 2 ถ้าใช้ส่วนประกอบโครงสร้าง 5 × 5 ทำการเปิดภาพรอยร้าวในตัวอย่างที่ 6.1 (`crack_mask`) จะเกิดอะไรขึ้น ข้อนี้บอกอะไรเกี่ยวกับการเลือกขนาดส่วนประกอบโครงสร้างในงานตรวจรอยร้าว
3. ในข้อ 3–4 ทำไม AP ก่อน NMS จึงต่ำกว่าหลัง NMS ทั้งที่ทั้งสองกรณีมีกรอบที่ถูกต้องชุดเดียวกัน และทำไม AP@0.75 จึงต่ำกว่า AP@0.5
4. ในส่วนที่ 5 ส่วนหลักที่แช่แข็งได้ผลดีกว่าพิกเซลดิบมากทั้งที่ฝึกพารามิเตอร์น้อยกว่า อธิบายด้วยแนวคิด "การใช้ค่าน้ำหนักร่วมกัน" และ "การสมมูลภายใต้การเลื่อน" และถ้าจะนำแนวทางนี้ไปใช้กับภาพลูกถ้วยจริงจากโดรน คุณจะแบ่งข้อมูลฝึก/ทดสอบอย่างไร

*คำตอบ:* (เขียนคำตอบที่นี่)

## สรุป

- **ภาพดิจิทัล** คืออาร์เรย์ NumPy เข้าถึงด้วย `F[r, c]` แต่ OpenCV วาดรูปด้วย $(x, y) = (c, r)$ และ `cv2.resize` รับ (กว้าง, สูง) ค่าเฉลี่ยของบริเวณรอยร้าว (157.3) ไม่บอกว่ามีรอยร้าว ต้องดูรูปแบบเชิงพื้นที่ GSD ของกล้องโดรนที่ 10 m คือ 0.833 mm/px และต้องบินไม่เกิน 8 m ถ้าต้องการให้รอยร้าว 2 mm กว้าง 3 พิกเซล (ส่วนที่ 1)
- **ปริภูมิสี** OpenCV เก็บภาพเป็น BGR และ H เป็น 0–179 กฎใน HSV ตรวจสนิมได้ทั้งในแดดและในเงา ขณะที่กฎใน RGB พลาดสนิมในเงา (ส่วนที่ 1.3)
- **การกรองภาพ** ฟังก์ชัน `filter2d` ที่เขียนตามรหัสเทียมให้ผลเท่ากับ `cv2.filter2D` `cv2.blur` `cv2.GaussianBlur` เมื่อเลือกการเติมขอบให้ตรงกัน ตัวกรองมัธยฐานกำจัดจุดรบกวนได้ (255 → 100) ส่วนตัวกรองค่าเฉลี่ยกระจายมันออกไป (≈ 117) และเคอร์เนลใหญ่ทำให้รอยร้าวจาง (ส่วนที่ 2)
- **การหาขอบ** Sobel ให้ $G_x = \mp 560$ ขนาบรอยร้าวแนวตั้ง ค่าเกณฑ์ของ Canny แลกเปลี่ยนระหว่างจุดรบกวนกับรอยร้าวที่ขาด และฮิสเทอรีซิสเก็บ P1–P3, P5–P6 โดยตัด Q ทิ้ง (ส่วนที่ 3)
- **Otsu + สัณฐานวิทยา + องค์ประกอบเชื่อมต่อ** แปลงภาพหน้าจอมิเตอร์เป็นตัวเลขแยกชิ้นได้ครบ การเปิดลบฝุ่น การปิดเชื่อมรอยขาด และแบบฝึกปฏิบัติข้อ 1–2 เขียน Otsu และการกร่อน/พองตัวเองได้ผลตรงกับ OpenCV (ส่วนที่ 4)
- **การจำแนกภาพ** คุณลักษณะเดียวอย่างความหนาแน่นของขอบทำนายภาพคราบสกปรกว่า "แตก" เกือบทั้งหมด ส่วน "ส่วนหลักที่แช่แข็ง + ส่วนหัว" ได้ผลดีที่สุดด้วยพารามิเตอร์ที่ฝึกเพียงหลักร้อย ซึ่งเป็นแนวคิดเดียวกับการสกัดคุณลักษณะของ ResNet-18 ที่ฝึกเพียง 1,539 จาก 11.2 ล้านพารามิเตอร์ (ส่วนที่ 5)
- **การตรวจจับวัตถุ** IoU ของลูกถ้วยในตัวอย่างที่ 6.14 คือ 0.566 วัตถุเล็กไวต่อความคลาดเคลื่อนของตำแหน่งมาก ตัวตรวจจับให้กรอบซ้ำซึ่งต้องกำจัดด้วย NMS ก่อนวัดผลด้วย AP (ตัวอย่างที่ 6.17 ได้ 0.683) **การแบ่งส่วนภาพ** ต้องรายงาน IoU/Dice ไม่ใช่ความถูกต้องรายพิกเซล (ส่วนที่ 6, แบบฝึกปฏิบัติข้อ 3–4)
- **การประยุกต์** เกณฑ์ $\mu + 3\sigma$ ของระบบตรวจหาค่าผิดปกติคือ 0.288 อัตราฐานต่ำทำให้ความแม่นยำของการคัดออกเหลือ 0.323 และภาพถ่ายความร้อนต้องวิเคราะห์จากอุณหภูมิดิบ พร้อมแยก "ร้อนเพราะโหลด" ออกจาก "ร้อนเพราะจุดต่อเสีย" (ส่วนที่ 7)

**ก้าวต่อไป:** บทที่ 7 และปฏิบัติการที่ 13 จะเปลี่ยนจากภาพไปสู่ข้อความ ซึ่งใช้แนวคิดการเรียนรู้แบบถ่ายโอนแบบเดียวกันกับโมเดลภาษาขนาดใหญ่

## เอกสารอ้างอิง

- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications* (2nd ed.). Springer. บทที่ 3, 5, 6
- Gonzalez, R. C., & Woods, R. E. (2018). *Digital Image Processing* (4th ed.). Pearson.
- Bradski, G. (2000). The OpenCV library. *Dr. Dobb's Journal of Software Tools, 25*(11), 120–125. เอกสารประกอบ: https://docs.opencv.org/
- Otsu, N. (1979). A threshold selection method from gray-level histograms. *IEEE Transactions on Systems, Man, and Cybernetics, 9*(1), 62–66.
- Canny, J. (1986). A computational approach to edge detection. *IEEE Transactions on Pattern Analysis and Machine Intelligence, 8*(6), 679–698.
- Redmon, J., Divvala, S., Girshick, R., & Farhadi, A. (2016). You only look once: Unified, real-time object detection. *Proceedings of CVPR*, 779–788.
- Everingham, M., et al. (2010). The PASCAL Visual Object Classes (VOC) challenge. *International Journal of Computer Vision, 88*(2), 303–338.
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *Proceedings of CVPR*, 770–778. (โครงสร้าง ResNet-18 ที่ใช้นับพารามิเตอร์)
- Deitsch, S., et al. (2019). Automatic classification of defective photovoltaic module cells in electroluminescence images. *Solar Energy, 185*, 455–468. ชุดข้อมูล ELPV: https://github.com/zae-bayern/elpv-dataset
- ภาพทั้งหมดในส่วนหลักของ notebook เป็นภาพจำลองที่สร้างด้วยโค้ด (seed = 42) ค่าตัวเลขของตัวอย่างเป็นค่าสมมติตามบทที่ 6 ของหนังสือ ภาพตัวอย่างในส่วนทางเลือก 6.5 คือ `skimage.data.astronaut()` (ภาพสาธารณสมบัติจาก NASA)